In [1]:
!pip install selenium webdriver-manager pandas

In [4]:
import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ==================== KONFIGURASI ====================

SEARCH_QUERIES = [
    "coffee shop Semarang Tengah",
    "coffee shop Semarang Barat",
    "coffee shop Semarang Timur",
    "coffee shop Semarang Selatan",
    "coffee shop Semarang Utara",
    "coffee shop Banyumanik Semarang",
    "coffee shop Tembalang Semarang",
    "coffee shop Ngaliyan Semarang",
]

MAX_SCROLL_PER_QUERY = 15
MAX_REVIEWS_PER_CAFE = 500  # target ulasan per cafe. Kalau cafe punya ulasan < 500, otomatis diambil semua (berhenti karena stagnan)
MAX_SCROLL_ATTEMPTS_REVIEW = 400  # jaring pengaman jumlah percobaan scroll
STAGNANT_LIMIT = 6  # berhenti kalau sekian kali scroll berturut-turut tidak menambah ulasan baru
MAX_SECONDS_PER_CAFE = 600  # jaring pengaman waktu: maksimal 10 menit per cafe

CAFE_FILE = "scraping_file_cafe_lengkap.csv"
REVIEWS_FILE = "scraping_file_google_maps_reviews.csv"


# ==================== SETUP ====================

def setup_driver():
    options = Options()
    options.add_argument("--lang=id-ID")
    options.add_argument("--disable-notifications")
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)


def handle_consent_dialog(driver):
    possible_texts = ["Terima semua", "Accept all", "Setuju", "I agree", "Saya setuju"]
    for text in possible_texts:
        try:
            btn = driver.find_element(By.XPATH, f"//button[contains(., '{text}')]")
            btn.click()
            print(f"  [i] Dialog consent ditemukan dan diklik ('{text}')")
            time.sleep(1.5)
            return True
        except Exception:
            continue
    return False


# ==================== TAHAP 1: Kumpulkan daftar cafe ====================

def search_cafes(driver, query):
    print(f"\nMencari area: {query}")
    url = f"https://www.google.com/maps/search/{query.replace(' ', '+')}"
    driver.get(url)
    time.sleep(random.uniform(3, 5))
    handle_consent_dialog(driver)

    try:
        feed = driver.find_element(By.CSS_SELECTOR, "div[role='feed']")
    except Exception:
        print("  [!] Panel hasil pencarian tidak ditemukan, skip.")
        return []

    for _ in range(MAX_SCROLL_PER_QUERY):
        driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", feed)
        time.sleep(random.uniform(1.5, 2.5))

    result_links = driver.find_elements(By.CSS_SELECTOR, "a.hfpxzc")
    cafes = []
    for el in result_links:
        nama = el.get_attribute("aria-label")
        link = el.get_attribute("href")
        if nama and link:
            cafes.append({"nama_cafe": nama, "link": link})

    print(f"  -> ditemukan {len(cafes)} cafe di area ini")
    return cafes


def collect_all_cafe_links(driver):
    all_cafes = {}
    for query in SEARCH_QUERIES:
        for c in search_cafes(driver, query):
            all_cafes[c["link"]] = c["nama_cafe"]
        time.sleep(random.uniform(3, 6))

    print(f"\nTotal cafe unik ditemukan: {len(all_cafes)}")
    return all_cafes


# ==================== TAHAP 2: Rating, price level, alamat, jam operasional ====================

def extract_cafe_summary(driver):
    summary = {
        "rating_rata2": None, "jumlah_ulasan_total": None,
        "harga_indikasi": None, "alamat": None, "jam_operasional": None
    }

    try:
        rating_el = driver.find_element(By.CSS_SELECTOR, "div.F7nice span[aria-hidden='true']")
        summary["rating_rata2"] = rating_el.text.strip()
    except Exception:
        print("  [!] rating_rata2 tidak ditemukan")

    try:
        jumlah_el = driver.find_element(
            By.CSS_SELECTOR,
            "div.F7nice span[aria-label*='ulasan'], div.F7nice span[aria-label*='review']"
        )
        angka = re.sub(r"[^\d]", "", jumlah_el.get_attribute("aria-label") or jumlah_el.text)
        summary["jumlah_ulasan_total"] = angka if angka else None
    except Exception:
        print("  [!] jumlah_ulasan_total tidak ditemukan")

    try:
        price_el = driver.find_elements(By.XPATH, "//span[contains(text(),'Rp') or contains(text(),'$')]")
        if price_el:
            summary["harga_indikasi"] = price_el[0].text.strip()
    except Exception:
        pass

    try:
        alamat_el = driver.find_element(By.XPATH, "//button[contains(@aria-label, 'Alamat')]")
        label = alamat_el.get_attribute("aria-label") or ""
        summary["alamat"] = label.replace("Alamat: ", "").strip()
    except Exception:
        print("  [!] alamat tidak ditemukan")

    jam_ditemukan = False
    hari_list = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]
    for hari in hari_list:
        try:
            found = driver.find_elements(By.XPATH, f"//*[contains(@aria-label, '{hari}')]")
            if found:
                summary["jam_operasional"] = (found[0].get_attribute("aria-label") or "").strip()
                jam_ditemukan = True
                break
        except Exception:
            continue

    if not jam_ditemukan:
        try:
            found = driver.find_elements(
                By.XPATH,
                "//*[contains(text(),'Buka') or contains(text(),'Tutup') or contains(text(),'24 jam')]"
            )
            if found:
                teks = found[0].text.strip()
                if teks:
                    summary["jam_operasional"] = teks
                    jam_ditemukan = True
        except Exception:
            pass

    if not jam_ditemukan:
        print("  [!] jam_operasional tidak ditemukan")

    return summary


# ==================== TAHAP 3: Ulasan + rating per ulasan ====================

def click_review_tab(driver, timeout=8):
    try:
        tab = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((
                By.XPATH,
                "//button[contains(@aria-label, 'Ulasan') or contains(@aria-label, 'Reviews')]"
            ))
        )
        tab.click()
        print("  [i] Tab ulasan diklik lewat STRATEGI 1 (aria-label)")
        return True
    except Exception as e:
        print(f"  [debug] Strategi 1 gagal: {type(e).__name__}")

    try:
        tabs = driver.find_elements(By.CSS_SELECTOR, "button[role='tab']")
        print(f"  [debug] Ditemukan {len(tabs)} tombol role='tab': "
              f"{[t.get_attribute('aria-label') for t in tabs]}")
        if len(tabs) >= 2:
            tabs[1].click()
            print("  [i] Tab ulasan diklik lewat STRATEGI 2 (role='tab' index ke-2)")
            return True
    except Exception as e:
        print(f"  [debug] Strategi 2 gagal: {type(e).__name__}: {e}")

    print("  [!] Tab ulasan GAGAL diklik (kedua strategi gagal)")
    return False


def find_scrollable_review_panel(driver):
    """Cari container ulasan yang BENAR-BENAR scrollable (scrollHeight > clientHeight),
    lalu scroll pakai scrollTop langsung -- pola yang sama persis dengan yang sudah
    terbukti jalan di search_cafes() untuk scroll daftar cafe. Ini jauh lebih reliable
    dibanding simulasi wheel/keyboard yang tergantung posisi elemen di viewport."""
    candidate_selectors = [
        "div.m6QErb[aria-label]",
        "div[role='main'] div.m6QErb",
        "div.DxyBCb",
    ]
    for sel in candidate_selectors:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            for el in els:
                is_scrollable = driver.execute_script(
                    "return arguments[0].scrollHeight > arguments[0].clientHeight + 10;", el
                )
                if is_scrollable:
                    return el
        except Exception:
            continue
    return None


def scroll_reviews_via_panel(driver, target_count):
    """Scroll panel ulasan pakai scrollTop, berhenti kalau:
    - sudah mencapai target_count ulasan, ATAU
    - stagnan (tidak nambah lagi -> berarti cafe ini ulasannya memang < target), ATAU
    - sudah melebihi batas jumlah percobaan / waktu per cafe (jaring pengaman)."""
    panel = find_scrollable_review_panel(driver)
    if panel is None:
        print("    [!] Panel ulasan scrollable tidak ditemukan! Coba ambil apa yang sudah termuat.")
        return driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")

    h = driver.execute_script("return arguments[0].scrollHeight;", panel)
    c = driver.execute_script("return arguments[0].clientHeight;", panel)
    print(f"    [debug] Panel ulasan ditemukan. scrollHeight={h}, clientHeight={c}")

    stagnant_count = 0
    last_count = 0
    start_time = time.time()

    for attempt in range(MAX_SCROLL_ATTEMPTS_REVIEW):
        elapsed = time.time() - start_time
        if elapsed > MAX_SECONDS_PER_CAFE:
            print(f"    -> berhenti, sudah {elapsed:.0f} detik di cafe ini (melebihi batas {MAX_SECONDS_PER_CAFE} detik)")
            break

        driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", panel)
        time.sleep(random.uniform(1.2, 2.0))

        current_reviews = driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")
        current_count = len(current_reviews)
        print(f"    scroll #{attempt + 1}: {current_count} ulasan termuat ({elapsed:.0f}s)")

        if target_count is not None and current_count >= target_count:
            print(f"    -> berhenti, target {target_count} ulasan tercapai ({current_count} termuat)")
            break

        if current_count == last_count:
            stagnant_count += 1
            if stagnant_count >= STAGNANT_LIMIT:
                print(f"    -> berhenti, tidak ada ulasan baru setelah {STAGNANT_LIMIT}x scroll "
                      f"(total: {current_count} ulasan -- kemungkinan ini semua ulasan yang tersedia)")
                break
        else:
            stagnant_count = 0

        last_count = current_count

    return driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")


def scrape_reviews_from_link(driver, nama_cafe, link):
    print(f"\nMembuka: {nama_cafe}")
    driver.get(link)

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h1"))
        )
    except Exception:
        print("  [!] Halaman cafe tidak termuat sempurna dalam 10 detik, tetap lanjut coba.")

    handle_consent_dialog(driver)

    summary = extract_cafe_summary(driver)
    print(f"  Rating: {summary['rating_rata2']}, Jumlah ulasan: {summary['jumlah_ulasan_total']}, "
          f"Harga: {summary['harga_indikasi']}, Jam: {summary['jam_operasional']}")

    if not click_review_tab(driver):
        return summary, []

    time.sleep(random.uniform(1.5, 2.5))

    initial_check = driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")
    print(f"  [debug] Segera setelah klik tab: {len(initial_check)} elemen div.jJc9Ad langsung ditemukan")

    review_blocks = scroll_reviews_via_panel(driver, MAX_REVIEWS_PER_CAFE)

    # Kalau termuat lebih banyak dari target (bisa terjadi karena batch load Google
    # tidak presisi per-item), potong ke target supaya konsisten per cafe.
    if MAX_REVIEWS_PER_CAFE is not None and len(review_blocks) > MAX_REVIEWS_PER_CAFE:
        review_blocks = review_blocks[:MAX_REVIEWS_PER_CAFE]

    reviews = []
    for block in review_blocks:
        try:
            teks_el = block.find_element(By.CSS_SELECTOR, "span.wiI7pd")
            teks = teks_el.text.strip()
        except Exception:
            teks = None

        try:
            rating_el = block.find_element(By.CSS_SELECTOR, "span[role='img']")
            rating_label = rating_el.get_attribute("aria-label") or ""
            angka_rating = re.search(r"[\d.,]+", rating_label)
            rating_bintang = angka_rating.group().replace(",", ".") if angka_rating else None
        except Exception:
            rating_bintang = None

        if teks:
            reviews.append({"ulasan": teks, "rating_bintang": rating_bintang})

    print(f"  -> berhasil ambil {len(reviews)} ulasan (target: {MAX_REVIEWS_PER_CAFE})")
    return summary, reviews


# ==================== MAIN ====================

def main():
    driver = setup_driver()
    cafe_rows = []
    review_rows = []

    try:
        driver.get("https://www.google.com/maps")
        time.sleep(2)
        handle_consent_dialog(driver)

        print("=== TAHAP 1: Mengumpulkan daftar cafe ===")
        cafe_links = collect_all_cafe_links(driver)

        print("\n=== TAHAP 2 & 3: Rating, price level, alamat, jam, dan ulasan tiap cafe ===")
        for i, (link, nama_cafe) in enumerate(cafe_links.items()):
            summary, reviews = scrape_reviews_from_link(driver, nama_cafe, link)

            cafe_rows.append({
                "nama_cafe": nama_cafe,
                "link": link,
                "alamat": summary["alamat"],
                "rating_rata2": summary["rating_rata2"],
                "jumlah_ulasan_total": summary["jumlah_ulasan_total"],
                "harga_indikasi": summary["harga_indikasi"],
                "jam_operasional": summary["jam_operasional"],
            })

            for r in reviews:
                review_rows.append({
                    "nama_cafe": nama_cafe,
                    "ulasan": r["ulasan"],
                    "rating_bintang": r["rating_bintang"],
                })

            jeda = random.uniform(8, 15)
            print(f"  Jeda {jeda:.1f} detik... ({i + 1}/{len(cafe_links)})")
            time.sleep(jeda)
    finally:
        driver.quit()

    df_cafe = pd.DataFrame(cafe_rows)
    df_cafe.to_csv(CAFE_FILE, index=False, encoding="utf-8-sig")

    df_reviews = pd.DataFrame(review_rows)
    df_reviews.to_csv(REVIEWS_FILE, index=False, encoding="utf-8-sig")

    print(f"\n=== SELESAI ===")
    print(f"Data cafe: {len(df_cafe)} -> {CAFE_FILE}")
    print(f"Data ulasan: {len(df_reviews)} -> {REVIEWS_FILE}")

    return df_cafe, df_reviews


df_cafe, df_reviews = main()

=== TAHAP 1: Mengumpulkan daftar cafe ===

Mencari area: coffee shop Semarang Tengah
  -> ditemukan 80 cafe di area ini

Mencari area: coffee shop Semarang Barat
  -> ditemukan 80 cafe di area ini

Mencari area: coffee shop Semarang Timur
  -> ditemukan 80 cafe di area ini

Mencari area: coffee shop Semarang Selatan
  -> ditemukan 80 cafe di area ini

Mencari area: coffee shop Semarang Utara
  -> ditemukan 80 cafe di area ini

Mencari area: coffee shop Banyumanik Semarang
  -> ditemukan 70 cafe di area ini

Mencari area: coffee shop Tembalang Semarang
  -> ditemukan 80 cafe di area ini

Mencari area: coffee shop Ngaliyan Semarang
  -> ditemukan 70 cafe di area ini

Total cafe unik ditemukan: 446

=== TAHAP 2 & 3: Rating, price level, alamat, jam, dan ulasan tiap cafe ===

Membuka: Eastman Coffee House
  Rating: 4,7, Jumlah ulasan: 4396, Harga: Rp 50–100 rb, Jam: Senin,07.00 hingga 23.00, Salin jam buka
  [i] Tab ulasan diklik lewat STRATEGI 1 (aria-label)
  [debug] Segera setelah klik 

In [1]:
pip install pandas rapidfuzz --break-system-packages

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 2.6 MB/s eta 0:00:01
   ----------------------------- ---------- 1.3/1.8 MB 2.2 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 2.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
SCRAPING ULANG - khusus cafe dengan cakupan ulasan rendah/gagal total.

Membaca daftar target dari cek_cakupan_ulasan.csv (hasil validasi sebelumnya),
memilih cafe dengan rating_rata2 kosong (gagal total) ATAU persentase_cakupan < 10%,
lalu scraping ulang KHUSUS cafe-cafe itu saja -- bukan full 440 cafe lagi.

PERBAIKAN dari versi sebelumnya:
- keyboard_scroll() sekarang klik ke TOMBOL TAB ULASAN sebagai titik fokus,
  BUKAN ke blok ulasan itu sendiri -- karena blok ulasan berisi tombol link ke
  profil reviewer yang bisa ke-klik tidak sengaja dan mengganggu proses loading.

CARA PAKAI:
1. Pastikan cek_cakupan_ulasan.csv ada di folder yang sama
2. pip install selenium webdriver-manager pandas
3. Jalankan: python retry_scrape.py
4. Hasil: retry_cafe.csv dan retry_reviews.csv
"""

import time
import random
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.actions.wheel_input import ScrollOrigin
from webdriver_manager.chrome import ChromeDriverManager

CEK_FILE = "cek_cakupan_ulasan.csv"
RETRY_CAFE_FILE = "retry_cafe.csv"
RETRY_REVIEWS_FILE = "retry_reviews.csv"

MAX_REVIEWS_PER_CAFE = 500
MAX_SCROLL_ATTEMPTS_REVIEW = 400
STAGNANT_LIMIT = 6
MAX_SECONDS_PER_CAFE = 900


# ==================== SETUP ====================

def setup_driver():
    options = Options()
    options.add_argument("--lang=id-ID")
    options.add_argument("--disable-notifications")
    # PENTING: sembunyikan fingerprint otomasi -- Google kemungkinan mendeteksi
    # navigator.webdriver=true dan menyajikan UI berbeda (versi dengan tombol
    # "muat lebih banyak" alih-alih infinite scroll) ke browser yang terdeteksi bot.
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("--window-size=1366,900")  # ukuran window wajar, bukan default aneh

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)

    # Timpa navigator.webdriver jadi undefined SEBELUM halaman apa pun dimuat
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    return driver


def handle_consent_dialog(driver):
    possible_texts = ["Terima semua", "Accept all", "Setuju", "I agree", "Saya setuju"]
    for text in possible_texts:
        try:
            btn = driver.find_element(By.XPATH, f"//button[contains(., '{text}')]")
            btn.click()
            print(f"  [i] Dialog consent ditemukan dan diklik ('{text}')")
            time.sleep(1.5)
            return True
        except Exception:
            continue
    return False


# ==================== RATING, PRICE, ALAMAT, JAM ====================

def extract_cafe_summary(driver):
    summary = {
        "rating_rata2": None, "jumlah_ulasan_total": None,
        "harga_indikasi": None, "alamat": None, "jam_operasional": None
    }
    try:
        rating_el = driver.find_element(By.CSS_SELECTOR, "div.F7nice span[aria-hidden='true']")
        summary["rating_rata2"] = rating_el.text.strip()
    except Exception:
        print("  [!] rating_rata2 tidak ditemukan")

    try:
        jumlah_el = driver.find_element(
            By.CSS_SELECTOR,
            "div.F7nice span[aria-label*='ulasan'], div.F7nice span[aria-label*='review']"
        )
        angka = re.sub(r"[^\d]", "", jumlah_el.get_attribute("aria-label") or jumlah_el.text)
        summary["jumlah_ulasan_total"] = angka if angka else None
    except Exception:
        print("  [!] jumlah_ulasan_total tidak ditemukan")

    try:
        price_el = driver.find_elements(By.XPATH, "//span[contains(text(),'Rp') or contains(text(),'$')]")
        if price_el:
            summary["harga_indikasi"] = price_el[0].text.strip()
    except Exception:
        pass

    try:
        alamat_el = driver.find_element(By.XPATH, "//button[contains(@aria-label, 'Alamat')]")
        label = alamat_el.get_attribute("aria-label") or ""
        summary["alamat"] = label.replace("Alamat: ", "").strip()
    except Exception:
        print("  [!] alamat tidak ditemukan")

    jam_ditemukan = False
    hari_list = ["Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"]
    for hari in hari_list:
        try:
            found = driver.find_elements(By.XPATH, f"//*[contains(@aria-label, '{hari}')]")
            if found:
                summary["jam_operasional"] = (found[0].get_attribute("aria-label") or "").strip()
                jam_ditemukan = True
                break
        except Exception:
            continue
    if not jam_ditemukan:
        try:
            found = driver.find_elements(
                By.XPATH,
                "//*[contains(text(),'Buka') or contains(text(),'Tutup') or contains(text(),'24 jam')]"
            )
            if found and found[0].text.strip():
                summary["jam_operasional"] = found[0].text.strip()
        except Exception:
            pass

    return summary


# ==================== ULASAN (dengan perbaikan keyboard_scroll) ====================

def click_review_tab(driver, timeout=8):
    """Sekarang MENGEMBALIKAN elemen tab-nya juga, supaya bisa dipakai lagi
    sebagai titik fokus keyboard yang AMAN (bukan blok ulasan)."""
    try:
        tab = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((
                By.XPATH,
                "//button[contains(@aria-label, 'Ulasan') or contains(@aria-label, 'Reviews')]"
            ))
        )
        tab.click()
        print("  [i] Tab ulasan diklik lewat STRATEGI 1 (aria-label)")
        return True, tab
    except Exception as e:
        print(f"  [debug] Strategi 1 gagal: {type(e).__name__}")

    try:
        tabs = driver.find_elements(By.CSS_SELECTOR, "button[role='tab']")
        if len(tabs) >= 2:
            tabs[1].click()
            print("  [i] Tab ulasan diklik lewat STRATEGI 2 (role='tab' index ke-2)")
            return True, tabs[1]
    except Exception as e:
        print(f"  [debug] Strategi 2 gagal: {type(e).__name__}: {e}")

    print("  [!] Tab ulasan GAGAL diklik (kedua strategi gagal)")
    return False, None


def cdp_wheel_scroll(driver, anchor_element, delta_y=700, ticks=3):
    """Teknik BARU: kirim event wheel lewat Chrome DevTools Protocol (CDP)
    langsung ke koordinat piksel di layar -- ini lebih 'rendah level' dibanding
    ActionChains Selenium biasa, dan tidak bergantung pada elemen mana yang
    'dianggap' scrollable oleh WebDriver. Dipakai kalau wheel & keyboard biasa
    sama-sama gagal."""
    try:
        if anchor_element is not None:
            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center', behavior: 'instant'});",
                anchor_element
            )
            time.sleep(0.3)
            rect = anchor_element.rect
            x = rect["x"] + rect["width"] / 2
            y = rect["y"] + rect["height"] / 2
        else:
            x, y = 400, 400

        for _ in range(ticks):
            driver.execute_cdp_cmd("Input.dispatchMouseEvent", {
                "type": "mouseWheel",
                "x": x, "y": y,
                "deltaX": 0, "deltaY": delta_y,
            })
            time.sleep(random.uniform(0.4, 0.7))
        return True
    except Exception as e:
        print(f"    [debug] cdp_wheel_scroll gagal: {type(e).__name__}: {e}")
        return False


def keyboard_scroll(driver, safe_focus_element, presses=4):
    """DIPERBAIKI: klik ke tombol TAB ULASAN (aman, idempotent), BUKAN ke blok
    ulasan -- supaya tidak ada risiko ke-klik tombol profil reviewer di dalamnya
    yang bisa mengganggu proses loading ulasan berikutnya."""
    try:
        if safe_focus_element is not None:
            ActionChains(driver).move_to_element(safe_focus_element).click().perform()
        for _ in range(presses):
            ActionChains(driver).send_keys(Keys.PAGE_DOWN).perform()
            time.sleep(random.uniform(0.5, 0.8))
        return True
    except Exception as e:
        print(f"    [debug] keyboard_scroll gagal: {type(e).__name__}: {e}")
        return False


def real_wheel_scroll(driver, anchor_element, amount=400, ticks=3):
    try:
        if anchor_element is not None:
            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center', behavior: 'instant'});",
                anchor_element
            )
            time.sleep(0.3)
            origin = ScrollOrigin.from_element(anchor_element)
        else:
            origin = ScrollOrigin.from_viewport(400, 400)
        for _ in range(ticks):
            ActionChains(driver).scroll_from_origin(origin, 0, amount).perform()
            time.sleep(random.uniform(0.4, 0.7))
        return True
    except Exception as e:
        print(f"    [debug] real_wheel_scroll gagal: {type(e).__name__}: {e}")
        try:
            driver.execute_script("window.scrollBy(0, 400);")
        except Exception:
            pass
        return False


def click_load_more_if_present(driver):
    """DIPERBAIKI: kata kunci 'Lainnya' saja ternyata terlalu umum -- kena
    tombol 'baca selengkapnya' di DALAM satu ulasan (memperluas teks
    terpotong), bukan tombol muat ulasan baru. Sekarang pakai frasa lebih
    spesifik DAN mengecualikan tombol yang berada di dalam blok ulasan
    (div.jJc9Ad), supaya tidak salah sasaran lagi."""
    keywords = ["ulasan lainnya", "lebih banyak ulasan", "muat ulasan", "more reviews", "tampilkan ulasan"]
    for kw in keywords:
        try:
            btn = driver.find_element(
                By.XPATH,
                f"//button[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', "
                f"'abcdefghijklmnopqrstuvwxyz'), '{kw}')][not(ancestor::div[contains(@class,'jJc9Ad')])]"
            )
            btn.click()
            print(f"    [i] Tombol '{kw}' ditemukan dan diklik (di luar blok ulasan)")
            return True
        except Exception:
            continue
    return False


def scroll_reviews(driver, tab_element):
    """anchor untuk WHEEL tetap blok ulasan terakhir (posisi harus presisi),
    tapi anchor untuk KEYBOARD sekarang pakai tab_element (aman)."""
    stagnant_count = 0
    last_count = 0
    start_time = time.time()

    for attempt in range(MAX_SCROLL_ATTEMPTS_REVIEW):
        elapsed = time.time() - start_time
        if elapsed > MAX_SECONDS_PER_CAFE:
            print(f"    -> berhenti, sudah {elapsed:.0f} detik (melebihi batas {MAX_SECONDS_PER_CAFE}s)")
            break

        current_reviews = driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")
        wheel_anchor = current_reviews[-1] if current_reviews else None

        # Cek dulu apakah ada tombol "muat lebih banyak" -- kalau ada dan berhasil
        # diklik, itu jauh lebih pasti daripada scroll dalam bentuk apa pun.
        tombol_diklik = click_load_more_if_present(driver)

        if tombol_diklik:
            time.sleep(1.0)  # beri waktu render kalau memang tombol asli, lanjut tetap scroll di bawah
        if stagnant_count < 2:
            real_wheel_scroll(driver, wheel_anchor, amount=400, ticks=3)
        elif stagnant_count < 4:
            cdp_wheel_scroll(driver, wheel_anchor, delta_y=700, ticks=3)
        else:
            keyboard_scroll(driver, tab_element, presses=4)  # <-- pakai tab, bukan blok ulasan

        time.sleep(random.uniform(1.0, 1.8))

        current_reviews = driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")
        current_count = len(current_reviews)
        print(f"    scroll #{attempt+1}: {current_count} ulasan termuat ({elapsed:.0f}s)")

        if current_count >= MAX_REVIEWS_PER_CAFE:
            print(f"    -> berhenti, target {MAX_REVIEWS_PER_CAFE} tercapai")
            break

        if current_count == last_count:
            stagnant_count += 1
            if stagnant_count >= STAGNANT_LIMIT:
                print(f"    -> berhenti, tidak ada ulasan baru setelah {STAGNANT_LIMIT}x scroll")
                break
        else:
            stagnant_count = 0
        last_count = current_count

    return driver.find_elements(By.CSS_SELECTOR, "div.jJc9Ad")


def scrape_one_cafe(driver, nama_cafe, link):
    print(f"\nMembuka: {nama_cafe}")

    halaman_termuat = False
    for percobaan in range(2):  # coba 2x kalau gagal load di percobaan pertama
        driver.get(link)
        try:
            WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
            halaman_termuat = True
            break
        except Exception:
            print(f"  [!] Percobaan ke-{percobaan+1}: halaman tidak termuat dalam 20 detik.")
            print(f"  [debug] URL saat ini : {driver.current_url}")
            print(f"  [debug] Judul halaman: {driver.title}")
            if percobaan == 0:
                print("  [i] Mencoba reload sekali lagi...")
                time.sleep(3)

    if not halaman_termuat:
        # Upaya terakhir: link lama mungkin sudah "basi" di sisi Google, coba
        # cari ulang lewat NAMA (bukan link tersimpan) sebelum benar-benar menyerah.
        print("  [i] Mencoba cari ulang lewat nama (link lama mungkin sudah tidak valid)...")
        try:
            search_url = f"https://www.google.com/maps/search/{nama_cafe.replace(' ', '+')}+Semarang"
            driver.get(search_url)
            time.sleep(4)
            first_result = driver.find_elements(By.CSS_SELECTOR, "a.hfpxzc")
            if first_result:
                first_result[0].click()
                WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
                halaman_termuat = True
                print("  [i] Berhasil ditemukan lewat pencarian ulang nama.")
        except Exception as e:
            print(f"  [debug] Pencarian ulang lewat nama juga gagal: {type(e).__name__}")

    if not halaman_termuat:
        try:
            nama_file_aman = "".join(c if c.isalnum() else "_" for c in nama_cafe)[:50]
            screenshot_path = f"debug_gagal_{nama_file_aman}.png"
            driver.save_screenshot(screenshot_path)
            print(f"  [debug] Screenshot disimpan: {screenshot_path} (cek manual untuk lihat kenapa gagal)")
        except Exception:
            pass
        return {"rating_rata2": None, "jumlah_ulasan_total": None, "harga_indikasi": None,
                "alamat": None, "jam_operasional": None}, []

    handle_consent_dialog(driver)

    summary = extract_cafe_summary(driver)
    print(f"  Rating: {summary['rating_rata2']}, Jumlah ulasan: {summary['jumlah_ulasan_total']}, "
          f"Harga: {summary['harga_indikasi']}, Jam: {summary['jam_operasional']}")

    berhasil, tab_element = click_review_tab(driver)
    if not berhasil:
        return summary, []

    time.sleep(random.uniform(1.5, 2.5))
    review_blocks = scroll_reviews(driver, tab_element)

    reviews = []
    for block in review_blocks[:MAX_REVIEWS_PER_CAFE]:
        try:
            teks = block.find_element(By.CSS_SELECTOR, "span.wiI7pd").text.strip()
        except Exception:
            teks = None
        try:
            rating_el = block.find_element(By.CSS_SELECTOR, "span[role='img']")
            label = rating_el.get_attribute("aria-label") or ""
            m = re.search(r"[\d.,]+", label)
            rating_bintang = m.group().replace(",", ".") if m else None
        except Exception:
            rating_bintang = None
        if teks:
            reviews.append({"ulasan": teks, "rating_bintang": rating_bintang})

    print(f"  -> berhasil ambil {len(reviews)} ulasan")
    return summary, reviews


# ==================== MAIN ====================

def main():
    df_cek = pd.read_csv(CEK_FILE)
    target = df_cek[df_cek["rating_rata2"].isna() | (df_cek["persentase_cakupan"] < 10)]
    print(f"Total cafe yang akan di-scraping ulang: {len(target)}")
    print(target[["nama_cafe", "jumlah_ulasan_total", "ulasan_terambil"]].to_string())

    driver = setup_driver()
    cafe_rows = []
    review_rows = []

    try:
        driver.get("https://www.google.com/maps")
        time.sleep(2)
        handle_consent_dialog(driver)

        for i, row in enumerate(target.itertuples(), start=1):
            print(f"\n=== [{i}/{len(target)}] {row.nama_cafe} ===")
            summary, reviews = scrape_one_cafe(driver, row.nama_cafe, row.link)

            cafe_rows.append({
                "nama_cafe": row.nama_cafe,
                "link": row.link,
                "alamat": summary["alamat"],
                "rating_rata2": summary["rating_rata2"],
                "jumlah_ulasan_total": summary["jumlah_ulasan_total"],
                "harga_indikasi": summary["harga_indikasi"],
                "jam_operasional": summary["jam_operasional"],
            })
            for r in reviews:
                review_rows.append({
                    "nama_cafe": row.nama_cafe,
                    "ulasan": r["ulasan"],
                    "rating_bintang": r["rating_bintang"],
                })

            jeda = random.uniform(8, 15)
            print(f"  Jeda {jeda:.1f} detik...")
            time.sleep(jeda)
    finally:
        driver.quit()

    pd.DataFrame(cafe_rows).to_csv(RETRY_CAFE_FILE, index=False, encoding="utf-8-sig")
    pd.DataFrame(review_rows).to_csv(RETRY_REVIEWS_FILE, index=False, encoding="utf-8-sig")

    print(f"\n=== SELESAI ===")
    print(f"Cafe hasil scraping ulang   -> {RETRY_CAFE_FILE}")
    print(f"Ulasan hasil scraping ulang -> {RETRY_REVIEWS_FILE}")
    print("\nLangkah selanjutnya: jalankan merge_retry.py untuk menggabungkan hasil ini ke data utama.")


if __name__ == "__main__":
    main()

Total cafe yang akan di-scraping ulang: 19
                               nama_cafe  jumlah_ulasan_total  ulasan_terambil
56                     YiSan Coffee Shop                  NaN                0
125                       EL LOCO COFFEE                 15.0                0
161   Ideologist Coffee And Social Space               5254.0              500
168                            coffeshop                  1.0                0
170           Keizai Coffee & Garden MD3                530.0               30
174                         CERITA.COFFE                  NaN                0
188              Dapur Mama Cafe & Resto               8776.0              500
197               Pelangi cafe singosari              10523.0              500
233                        Lajoer Coffee                  2.0                0
344                          Tawa Coffee                  2.0                0
352                    Mbah Imam / Smith                  1.0                0
368      

In [3]:
"""
MERGE RETRY - Menggabungkan hasil scraping ulang (retry_cafe.csv, retry_reviews.csv)
ke data mentah utama (file_cafe_lengkap.csv, file_google_maps_reviews.csv).

Setelah ini selesai, jalankan ULANG step3_cleaning.py dan validasi_data.py dari awal
-- supaya data yang sudah digabung diproses lagi secara konsisten (dedup, cleaning),
bukan cuma ditambal manual.

CARA PAKAI:
Taruh file ini satu folder dengan file_cafe_lengkap.csv, file_google_maps_reviews.csv,
retry_cafe.csv, retry_reviews.csv. Jalankan: python merge_retry.py
"""

import pandas as pd

df_cafe_lama = pd.read_csv("scraping_file_cafe_lengkap.csv")
df_reviews_lama = pd.read_csv("scraping_file_google_maps_reviews.csv")
df_cafe_retry = pd.read_csv("retry_cafe.csv")
df_reviews_retry = pd.read_csv("retry_reviews.csv")

print(f"Data cafe lama       : {len(df_cafe_lama)}")
print(f"Data cafe hasil retry: {len(df_cafe_retry)}")

# --- Gabungkan data cafe: baris retry MENGGANTIKAN baris lama untuk nama_cafe yang sama ---
nama_di_retry = set(df_cafe_retry["nama_cafe"])
df_cafe_gabungan = df_cafe_lama[~df_cafe_lama["nama_cafe"].isin(nama_di_retry)]
df_cafe_gabungan = pd.concat([df_cafe_gabungan, df_cafe_retry], ignore_index=True)
print(f"Data cafe setelah digabung: {len(df_cafe_gabungan)} (harus sama dengan jumlah cafe lama)")

# --- Gabungkan data ulasan: buang ulasan lama milik cafe yang di-retry, ganti dengan yang baru ---
df_reviews_gabungan = df_reviews_lama[~df_reviews_lama["nama_cafe"].isin(nama_di_retry)]
df_reviews_gabungan = pd.concat([df_reviews_gabungan, df_reviews_retry], ignore_index=True)
print(f"Data ulasan setelah digabung: {len(df_reviews_gabungan)} (dari {len(df_reviews_lama)} lama)")

# --- Simpan, menimpa file mentah utama (backup dulu file lama secara manual kalau mau aman) ---
df_cafe_gabungan.to_csv("file_cafe_lengkap.csv", index=False, encoding="utf-8-sig")
df_reviews_gabungan.to_csv("file_google_maps_reviews.csv", index=False, encoding="utf-8-sig")

print("\n=== SELESAI ===")
print("file_cafe_lengkap.csv dan file_google_maps_reviews.csv sudah diperbarui.")
print("Langkah selanjutnya: jalankan ULANG step3_cleaning.py, lalu validasi_data.py.")

Data cafe lama       : 446
Data cafe hasil retry: 19
Data cafe setelah digabung: 446 (harus sama dengan jumlah cafe lama)
Data ulasan setelah digabung: 72915 (dari 72906 lama)

=== SELESAI ===
file_cafe_lengkap.csv dan file_google_maps_reviews.csv sudah diperbarui.
Langkah selanjutnya: jalankan ULANG step3_cleaning.py, lalu validasi_data.py.


In [1]:
"""
SCRIPT GABUNGAN - Menjalankan 3 tahap sekaligus secara berurutan:
1. Merge hasil retry_scrape ke data mentah utama (file_cafe_lengkap.csv, file_google_maps_reviews.csv)
2. Cleaning & deduplikasi + gabung atribut dari OSM (osm_raw.csv) -> cafe_clean.csv, reviews_clean.csv
3. Validasi kualitas data (cetak laporan + simpan cek_cakupan_ulasan.csv)

Perbaikan versi ini:
- Emoji diganti SPASI (bukan dihapus), supaya "enak😍yg" tidak menyambung jadi "enakyg".
- Kamus normalisasi diperluas + menangani huruf berulang ("ygg", "gaaak") dan
  kata ulang berangka ("orang2" -> "orang orang").
- Ada self-test normalisasi, laporan singkatan yang masih tersisa, dan laporan
  token pendek paling sering (untuk mencari singkatan baru secara iteratif).
- Enter/newline di dalam teks ulasan dirapikan jadi spasi (mencegah baris CSV "rusak").

CARA PAKAI:
Taruh semua file berikut di folder yang sama, lalu jalankan:
  - file_cafe_lengkap.csv
  - file_google_maps_reviews.csv
  - osm_raw.csv
  - retry_cafe.csv       (opsional -- kalau tidak ada, tahap ini dilewati)
  - retry_reviews.csv    (opsional -- kalau tidak ada, tahap ini dilewati)

python gabung_bersih_validasi.py
"""

import os
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from rapidfuzz.distance import OSA

CAFE_FILE = "file_cafe_lengkap.csv"
REVIEWS_FILE = "file_google_maps_reviews.csv"
OSM_FILE = "osm_raw.csv"
RETRY_CAFE_FILE = "retry_cafe.csv"
RETRY_REVIEWS_FILE = "retry_reviews.csv"

CAFE_CLEAN_OUT = "cafe_clean.csv"
REVIEWS_CLEAN_OUT = "reviews_clean.csv"
CEK_CAKUPAN_OUT = "cek_cakupan_ulasan.csv"
KANDIDAT_TYPO_OUT = "kandidat_typo.csv"

DEDUP_NAME_THRESHOLD = 90
OSM_MATCH_THRESHOLD = 85


# ==================== FUNGSI-FUNGSI BANTU ====================

def normalize_name(name):
    if pd.isna(name):
        return ""
    name = name.lower().strip()
    name = re.sub(r"[^\w\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def extract_lat_lng(link):
    if pd.isna(link):
        return None, None
    match = re.search(r"!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)", link)
    if match:
        return float(match.group(1)), float(match.group(2))
    return None, None


def haversine_km(lat1, lon1, lat2, lon2):
    from math import radians, sin, cos, sqrt, atan2
    R = 6371
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


# ---------- Emoji ----------
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticon wajah
    "\U0001F300-\U0001F5FF"  # simbol & piktograf
    "\U0001F680-\U0001F6FF"  # transportasi & peta
    "\U0001F1E0-\U0001F1FF"  # bendera
    "\U00002700-\U000027BF"  # dingbat
    "\U0001F900-\U0001F9FF"  # simbol tambahan
    "\U00002600-\U000026FF"  # simbol lain-lain
    "\U0001FA70-\U0001FAFF"  # simbol tambahan lanjutan
    "\U0001F004\U0001F0CF\U0001F18E"
    "\U0000FE0F"              # variation selector
    "\U0000200D"              # zero-width joiner
    "]+", flags=re.UNICODE
)


# Selain daftar rentang di atas, hapus SEMUA karakter berkategori Unicode "So"
# (Symbol, other: bintang, hati, panah, tanda centang, dll), blok emoji 1F000-1FAFF,
# blok 2B00-2BFF (mis. bintang U+2B50), dan penyambung emoji. Cara ini menangkap
# emoji/simbol yang tidak terdaftar di rentang manual.
_SIMBOL_DIPERTAHANKAN = "\u00b0"   # derajat (25°C) tetap dipertahankan


def _bangun_pola_simbol():
    rentang, mulai, akhir = [], None, None
    for cp in range(0x80, 0x110000):
        if 0xD800 <= cp <= 0xDFFF:
            cocok = False
        else:
            ch = chr(cp)
            cocok = (
                (unicodedata.category(ch) == "So" and ch not in _SIMBOL_DIPERTAHANKAN)
                or 0x1F000 <= cp <= 0x1FAFF
                or 0x2B00 <= cp <= 0x2BFF
                or 0xE0020 <= cp <= 0xE007F
                or cp in (0x200D, 0xFE0F, 0x20E3)
            )
        if cocok:
            if mulai is None:
                mulai = cp
            akhir = cp
        elif mulai is not None:
            rentang.append((mulai, akhir))
            mulai = None
    if mulai is not None:
        rentang.append((mulai, akhir))
    isi = "".join(
        re.escape(chr(a)) if a == b else re.escape(chr(a)) + "-" + re.escape(chr(b))
        for a, b in rentang
    )
    return re.compile("[" + isi + "]+")


POLA_SIMBOL = _bangun_pola_simbol()


def bersihkan_emoji(teks):
    # Emoji diganti SPASI (bukan ""), supaya kata di kiri-kanannya tidak menyambung.
    # \s+ juga merapikan newline/tab di dalam teks ulasan.
    teks = EMOJI_PATTERN.sub(" ", str(teks))
    teks = POLA_SIMBOL.sub(" ", teks)
    teks = re.sub(r"\s+", " ", teks).strip()
    return teks


# ---------- Normalisasi singkatan / kata tidak baku ----------
KAMUS_NORMALISASI = {
    # kata penghubung / umum
    "yg": "yang", "yng": "yang", "dgn": "dengan", "dg": "dengan", "dgan": "dengan",
    "utk": "untuk", "untk": "untuk", "dr": "dari", "krn": "karena", "karna": "karena",
    "krna": "karena", "sm": "sama", "tp": "tapi", "tpi": "tapi",
    "dll": "dan lain lain", "dsb": "dan sebagainya", "dst": "dan seterusnya",
    "pd": "pada", "spt": "seperti", "kyk": "seperti", "sbg": "sebagai",
    "ttg": "tentang", "tsb": "tersebut", "lbh": "lebih", "lbih": "lebih",
    "sll": "selalu", "slalu": "selalu", "spy": "supaya", "sblm": "sebelum",
    "stlh": "setelah", "stelah": "setelah", "smpe": "sampai", "sampe": "sampai",
    "msh": "masih", "masi": "masih", "hny": "hanya", "cm": "cuma", "cuman": "cuma",
    "plg": "paling", "dlm": "dalam", "jgn": "jangan", "bbrp": "beberapa",
    "n": "dan", "ni": "ini", "tu": "itu",
    # negasi
    "tdk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "ngga": "tidak",
    "gk": "tidak", "kaga": "tidak", "engga": "tidak", "enggak": "tidak",
    "ngak": "tidak", "gpp": "tidak apa apa", "gapapa": "tidak apa apa",
    "gada": "tidak ada", "blm": "belum", "blum": "belum", "bkn": "bukan",
    # kata ganti
    "sy": "saya", "gw": "saya", "gue": "saya", "gua": "saya",
    "kmu": "kamu", "lu": "kamu", "loe": "kamu",
    # waktu & keadaan
    "udh": "sudah", "udah": "sudah", "uda": "sudah", "sdh": "sudah", "dah": "sudah",
    "skrg": "sekarang", "sekrg": "sekarang", "kmrn": "kemarin", "kemaren": "kemarin",
    "bru": "baru", "lg": "lagi", "lgi": "lagi", "wkt": "waktu", "hbs": "habis",
    "jd": "jadi", "jdi": "jadi", "jg": "juga", "jga": "juga",
    "klo": "kalau", "kalo": "kalau", "kl": "kalau", "gt": "begitu", "gitu": "begitu",
    "gmn": "bagaimana", "gimana": "bagaimana", "knp": "kenapa",
    "aja": "saja", "aj": "saja", "ajah": "saja", "doang": "saja", "sj": "saja",
    # kata kerja / sifat
    "bs": "bisa", "bsa": "bisa", "hrs": "harus", "dpt": "dapat", "dpet": "dapat",
    "dtg": "datang", "liat": "lihat", "lht": "lihat", "pgn": "pengen", "pngen": "pengen",
    "pke": "pakai", "pake": "pakai", "pkai": "pakai", "tau": "tahu", "tw": "tahu",
    "trs": "terus", "trus": "terus", "mnrt": "menurut",
    "emg": "memang", "emng": "memang", "emang": "memang",
    "pdhl": "padahal", "padhal": "padahal", "sptnya": "sepertinya",
    "kayanya": "sepertinya", "kayaknya": "sepertinya", "krg": "kurang", "kurng": "kurang",
    "sndiri": "sendiri", "sndri": "sendiri", "bnr": "benar", "bner": "benar",
    "bgt": "banget", "bngt": "banget", "sgt": "sangat",
    "bgs": "bagus", "bgus": "bagus", "enk": "enak", "mantab": "mantap",
    "asik": "asyik", "cafe": "kafe",   # penyeragaman ejaan (hapus "cafe" kalau tidak mau diseragamkan)
    "mhl": "mahal", "mrh": "murah", "rame": "ramai", "ramee": "ramai",
    "lmyn": "lumayan", "sdkt": "sedikit", "bnyk": "banyak", "byk": "banyak",
    # nomina
    "org": "orang", "tmpt": "tempat", "tmpat": "tempat", "tmn": "teman", "temen": "teman",
    "hrg": "harga", "hrga": "harga", "mkn": "makan", "mknan": "makanan", "mkanan": "makanan",
    "mnm": "minum", "jln": "jalan", "jl": "jalan", "pel": "pelayanan",
    "dket": "dekat", "dkt": "dekat", "dpn": "depan", "blkg": "belakang",
    "kesini": "ke sini", "disini": "di sini", "kesana": "ke sana", "disana": "di sana",
    "pesen": "pesan", "dateng": "datang", "bener": "benar", "nunggu": "menunggu",
    "dapet": "dapat", "sekedar": "sekadar", "kopsu": "kopi susu",
    "aplg": "apalagi",
    # typo umum (tambahkan hasil tinjauan kandidat_typo.csv di sini)
    "wahib": "wajib", "snagat": "sangat",
    "makasih": "terima kasih", "mksh": "terima kasih", "trims": "terima kasih",
    "tks": "terima kasih", "thx": "terima kasih",
    "recomended": "recommended", "rekomended": "recommended", "recom": "recommended",
    "rekomen": "recommended", "reko": "recommended",
}


# Typo yang SUDAH ditinjau manual dari kandidat_typo.csv. Aturannya: hanya kata yang bukan kata sah
# (Indonesia/Inggris) dan punya satu perbaikan yang tidak ambigu. Kata sah seperti "walau", "tepat",
# "kalah", "taman" JANGAN dimasukkan, walau mirip kata lain, karena artinya akan berubah.
KAMUS_TYPO = {
    "tempatnyaa": "tempatnya", "nyamann": "nyaman", "pelayananya": "pelayanannya",
    "suasanya": "suasananya", "mayan": "lumayan", "sekalii": "sekali", "nugass": "nugas",
    "minumanya": "minumannya", "bangeet": "banget", "indor": "indoor", "tempatny":
    "tempatnya", "kopinyaa": "kopinya", "nongky": "nongki", "bnget": "banget", "bnyak":
    "banyak", "kureng": "kurang", "outdor": "outdoor", "mantaps": "mantap", "buanget":
    "banget", "cucok": "cocok", "nongkii": "nongki", "nongrong": "nongkrong", "nyamaan":
    "nyaman", "tempanya": "tempatnya", "pingin": "pengen", "mantaap": "mantap", "kerenn":
    "keren", "makanannyaa": "makanannya", "cofee": "coffee", "pelayanannyaa":
    "pelayanannya", "chiken": "chicken", "trnyata": "ternyata", "teruss": "terus", "skali":
    "sekali", "kereen": "keren", "trlalu": "terlalu", "ramaah": "ramah", "nongkrongg":
    "nongkrong", "temptnya": "tempatnya", "kluarga": "keluarga", "smoga": "semoga",
    "pkoknya": "pokoknya", "makanann": "makanan", "buanyak": "banyak", "ngbrol": "ngobrol",
    "harganyaa": "harganya", "sperti": "seperti", "makananny": "makanannya", "terjangkauu":
    "terjangkau", "ngopii": "ngopi", "tempate": "tempatnya", "banyaak": "banyak", "ngupi":
    "ngopi", "rasanyaa": "rasanya", "semuanyaa": "semuanya", "baget": "banget",
    "makannanya": "makanannya", "kopiny": "kopinya", "superr": "super", "bisaa": "bisa",
    "suasanannya": "suasananya", "instagramble": "instagramable", "kerjaa": "kerja",
    "sdikit": "sedikit", "pelayanann": "pelayanan", "hiden": "hidden", "mantaf": "mantap",
    "apalgi": "apalagi", "keluargaa": "keluarga", "banya": "banyak", "smpai": "sampai",
    "makannannya": "makanannya", "creamyy": "creamy", "outdornya": "outdoornya", "lumayann":
    "lumayan", "menunyaa": "menunya", "tempatmya": "tempatnya", "nonkrong": "nongkrong",
    "suasannya": "suasananya", "asyiik": "asyik", "trmpat": "tempat", "pdahal": "padahal",
    "kmren": "kemarin", "macha": "matcha", "bangt": "banget", "muantap": "mantap",
    "susananya": "suasananya", "cobainn": "cobain", "baristanyaa": "baristanya", "hargaa":
    "harga", "stengah": "setengah", "estetic": "estetik", "semarangg": "semarang",
}
KAMUS_NORMALISASI.update(KAMUS_TYPO)


# Kata yang SUDAH ditinjau dan diputuskan BUKAN typo (kata sah, kata asing, dialek, atau ambigu).
# Disimpan di sini supaya tidak muncul lagi di laporan kandidat typo.
KATA_DIABAIKAN = {
    "tepatnya", "jualan", "anget", "tenan", "orange", "serving", "variant", "rokoknya",
    "susana", "annya", "tempt", "snacks", "bernama", "datangi", "seberapa", "rasakan",
    "menurun", "bersantap", "temani", "tambahi", "merekah", "nerima", "services", "places",
    "pesto", "wcnya", "worthy", "superb", "cheesy", "plate", "psnya", "brasa", "krasa",
    "juarak", "pelayanya",
}


def _cari_token(low):
    """Cari 1 token (huruf kecil) di kamus, termasuk versi huruf berulang ("ygg", "gaaak")."""
    if low in KAMUS_NORMALISASI:
        return KAMUS_NORMALISASI[low]
    if len(low) >= 3:
        squeezed = re.sub(r"(.)\1+", r"\1", low)
        if squeezed in KAMUS_NORMALISASI:
            return KAMUS_NORMALISASI[squeezed]
    return None


def _cari_di_kamus(token):
    low = token.lower()
    hasil = _cari_token(low)
    if hasil is not None:
        return hasil
    # Token berakhiran "nya": "tmptnya" -> "tempatnya", "kayaknya" -> "sepertinya"
    if len(low) > 4 and low.endswith("nya"):
        hasil = _cari_token(low[:-3])
        if hasil is not None:
            return hasil + "nya"
    return None


# "nya" yang terpisah dari kata induknya oleh spasi / tanda baca:
# "enak nya", "Makanan.nya", "service'nya", "Mas\"nya", "\u201ckopsu\u201d nya"
_POLA_NYA = re.compile(r'''(?<=\w)[\s'"\u2019\u2018\u201c\u201d\-\u2013.+]+nya\b''', flags=re.IGNORECASE)


def normalisasi_singkatan(teks):
    teks = str(teks)
    # 1) kata ulang berangka: "orang2" -> "orang orang", "yg2" -> "yg yg" (dibakukan di langkah 4)
    teks = re.sub(r"\b([A-Za-z]+)2\b", r"\1 \1", teks)
    # 2) "t4" = "tempat"
    teks = re.sub(r"\bt4\b", "tempat", teks, flags=re.IGNORECASE)
    # "ny" = "nya" ("kopi v60 ny juga enak"); "d" = "di" ("d smg" = "di semarang")
    teks = re.sub(r"\bny\b", "nya", teks)
    teks = re.sub(r"(?<![\w'\u2019])d(?=\s+[a-z])", "di", teks)
    # "smg" = Semarang HANYA setelah kata depan/tempat ("di smg", "ke smg", "kota smg").
    # "smg" sendirian ambigu (bisa "semoga") sehingga tidak diubah.
    teks = re.sub(r"\b(di|ke|dari|kota|khas)\s+smg\b", r"\1 semarang", teks, flags=re.IGNORECASE)
    # 3) rapatkan "nya" SEBELUM kamus dipakai: "kayak nya" -> "kayaknya" -> "sepertinya"
    teks = _POLA_NYA.sub("nya", teks)

    # 4) ganti per token huruf; token yang tidak ada di kamus dibiarkan
    def ganti(m):
        hasil = _cari_di_kamus(m.group(0))
        return hasil if hasil is not None else m.group(0)

    teks = re.sub(r"[A-Za-z]+", ganti, teks)
    return re.sub(r"\s+", " ", teks).strip()


# ---------- Huruf memanjang ("enakkk", "bangettt", "yaaa") ----------
_RUN3 = re.compile(r"([a-z])\1{2,}")


def bangun_vocab(seri_teks):
    """Hitung frekuensi kata di seluruh ulasan (dipakai untuk memilih ejaan yang benar)."""
    vocab = Counter()
    for u in seri_teks:
        vocab.update(re.findall(r"[a-z]+", str(u).lower()))
    return vocab


def perbaiki_pemanjangan(teks, vocab):
    """
    Huruf berulang >= 3 kali dikecilkan. Pilihan 1 huruf ("goood" -> "god") atau
    2 huruf ("good") ditentukan dari mana yang lebih sering muncul di korpus.
    Huruf ganda di akhir kata (k/t/h/p) seperti "enakk" -> "enak" hanya diubah
    kalau bentuk tanpa gandanya jauh lebih sering (>= 10x), supaya kata asing
    yang memang berhuruf ganda tidak rusak.
    """
    def ganti(m):
        tok = m.group(0)
        low = tok.lower()
        baru = low
        if _RUN3.search(low):
            c1 = _RUN3.sub(r"\1", low)
            c2 = _RUN3.sub(r"\1\1", low)
            baru = c2 if vocab.get(c2, 0) > vocab.get(c1, 0) else c1
        if len(baru) >= 4 and baru[-1] == baru[-2] and baru[-1] in "kthp":
            stem = baru[:-1]
            if vocab.get(stem, 0) >= max(1, 10 * vocab.get(baru, 0)):
                baru = stem
        return tok if baru == low else baru

    return re.sub(r"[A-Za-z]+", ganti, str(teks))


def muat_kamus_kata_dasar():
    """Kata dasar bahasa Indonesia dari paket Sastrawi (opsional: pip install PySastrawi).
    Hanya dipakai untuk MENYARING laporan kandidat typo, tidak pernah untuk mengubah teks."""
    try:
        import Sastrawi
        path = os.path.join(os.path.dirname(Sastrawi.__file__), "Stemmer", "data", "kata-dasar.txt")
        with open(path, encoding="utf-8") as f:
            return {w.strip().lower() for w in f if w.strip()}
    except Exception:
        return set()


def cari_kandidat_typo(vocab, min_typo=2, maks_typo=200, min_benar=100, rasio=20, panjang_min=5,
                       kata_sah=frozenset()):
    """
    Cari kata jarang yang hanya beda 1 huruf (ganti/tambah/hapus/tukar posisi) dari kata
    yang JAUH lebih sering. Hasilnya hanya kandidat untuk ditinjau manual, tidak diubah
    otomatis, karena kata yang benar pun bisa mirip kata lain (mis. "makan" vs "makna").
    """
    benar = [w for w, c in vocab.items() if c >= min_benar and len(w) >= panjang_min]
    jarang = [(w, c) for w, c in vocab.items()
              if min_typo <= c <= maks_typo and len(w) >= panjang_min
              and w not in KAMUS_NORMALISASI and w not in kata_sah and w not in KATA_DIABAIKAN]
    hasil = []
    if not benar or not jarang:
        return hasil
    kata = [w for w, _ in jarang]
    for i in range(0, len(kata), 2000):
        blok = kata[i:i + 2000]
        # matriks jarak (dihitung paralel); nilai 1 = beda tepat 1 huruf
        mat = process.cdist(blok, benar, scorer=OSA.distance, score_cutoff=1,
                            workers=-1, dtype=np.uint8)
        for r, w in enumerate(blok):
            idx = np.nonzero(mat[r] == 1)[0]
            if len(idx) == 0:
                continue
            b, cb = max(((benar[j], vocab[benar[j]]) for j in idx), key=lambda x: x[1])
            c = vocab[w]
            if cb >= rasio * c:
                hasil.append((w, c, b, cb))
    hasil.sort(key=lambda x: -x[1])
    return hasil


def self_test_normalisasi():
    """Tes cepat supaya ketahuan langsung kalau fungsi normalisasi tidak bekerja."""
    vocab_uji = Counter({"banget": 100, "bangett": 5, "enak": 100, "enakk": 3,
                         "good": 50, "god": 1, "ya": 100, "yaa": 5})

    def alur(t):
        return normalisasi_singkatan(perbaiki_pemanjangan(bersihkan_emoji(t), vocab_uji))

    kasus = {
        "Cafe yg eatery nya sm nasi": "kafe yang eaterynya sama nasi",
        "tempatnya asik": "tempatnya asyik",
        "ygg enak, tp muter2 klo bgt": "yang enak, tapi muter muter kalau banget",
        "Pelayanan dr dpn good": "Pelayanan dari depan good",
        "kopi n sarapan": "kopi dan sarapan",
        "gaaak nyangka, emng udah beda": "tidak nyangka, memang sudah beda",
        "harga juga bagus": "harga juga bagus",   # kata baku tidak boleh berubah
        "kayak nya enak, kaya nya oke": "sepertinya enak, sepertinya oke",
        "tmpt nya bgs, t4 nyaman": "tempatnya bagus, tempat nyaman",
        "bangettt enaaak goood yaaa enakk": "banget enak good ya enak",
        "Makanan.nya enak, service'nya jelek": "Makanannya enak, servicenya jelek",
        'Mas"nya good, \u201ckopsu\u201d nya cremy': "Masnya good, \u201ckopi susunya cremy",
        "espresso.nya juga oke": "espressonya juga oke",
        "bintang 5 \u2b50 \u2b50 \u2605 \u2764\ufe0f \U0001f31f \u2728 mantap": "bintang 5 mantap",
        "suhu 25\u00b0C": "suhu 25\u00b0C",   # derajat tidak boleh hilang
        "kopi v60 ny juga enak": "kopi v60nya juga enak",
        "aplg buka d smg": "apalagi buka di semarang",
        "semoga smg tetap enak": "semoga smg tetap enak",   # smg sendirian tidak diubah
        "wahib coba, snagat enak": "wajib coba, sangat enak",
        "nyamann sekalii kerenn, mayan buat nongkrong": "nyaman sekali keren, lumayan buat nongkrong",
        "walau tepat kalah, taman kota": "walau tepat kalah, taman kota",   # kata sah tidak boleh berubah
        "coffee sama latte": "coffee sama latte",  # huruf ganda kata asing tidak boleh rusak
    }
    gagal = 0
    for asal, harapan in kasus.items():
        hasil = alur(asal)
        if hasil != harapan:
            gagal += 1
            print(f"  [GAGAL] '{asal}' -> '{hasil}' (harusnya '{harapan}')")
    print(f"Self-test normalisasi: {len(kasus) - gagal}/{len(kasus)} lulus")
    e = alur("enak😍yg bagus")
    print(f"Self-test emoji: 'enak😍yg bagus' -> '{e}' (harusnya 'enak yang bagus')")


# ==================== TAHAP 1: MERGE HASIL RETRY ====================

print("#" * 70)
print("TAHAP 1: MERGE HASIL RETRY KE DATA MENTAH UTAMA")
print("#" * 70)

df_cafe_raw = pd.read_csv(CAFE_FILE)
df_reviews_raw = pd.read_csv(REVIEWS_FILE)
print(f"Data cafe mentah (sebelum merge retry)   : {len(df_cafe_raw)}")
print(f"Data ulasan mentah (sebelum merge retry) : {len(df_reviews_raw)}")

if os.path.exists(RETRY_CAFE_FILE) and os.path.exists(RETRY_REVIEWS_FILE):
    df_cafe_retry = pd.read_csv(RETRY_CAFE_FILE)
    df_reviews_retry = pd.read_csv(RETRY_REVIEWS_FILE)
    print(f"Ditemukan hasil retry: {len(df_cafe_retry)} cafe, {len(df_reviews_retry)} ulasan")

    nama_di_retry = set(df_cafe_retry["nama_cafe"])

    # Baris lama untuk cafe yang di-retry DIGANTI, bukan ditambah dobel
    df_cafe_raw = df_cafe_raw[~df_cafe_raw["nama_cafe"].isin(nama_di_retry)]
    df_cafe_raw = pd.concat([df_cafe_raw, df_cafe_retry], ignore_index=True)

    df_reviews_raw = df_reviews_raw[~df_reviews_raw["nama_cafe"].isin(nama_di_retry)]
    df_reviews_raw = pd.concat([df_reviews_raw, df_reviews_retry], ignore_index=True)

    # Timpa file mentah utama supaya konsisten untuk run berikutnya
    df_cafe_raw.to_csv(CAFE_FILE, index=False, encoding="utf-8-sig")
    df_reviews_raw.to_csv(REVIEWS_FILE, index=False, encoding="utf-8-sig")

    print(f"Setelah merge retry -> cafe: {len(df_cafe_raw)}, ulasan: {len(df_reviews_raw)}")
    print(f"({CAFE_FILE} dan {REVIEWS_FILE} sudah diperbarui)")
else:
    print("Tidak ditemukan retry_cafe.csv / retry_reviews.csv -- tahap ini dilewati.")

df_cafe = df_cafe_raw.copy()
df_reviews = df_reviews_raw.copy()


# ==================== TAHAP 2: CLEANING & DEDUPLIKASI ====================

print("\n" + "#" * 70)
print("TAHAP 2: CLEANING, DEDUPLIKASI, DAN GABUNG ATRIBUT OSM")
print("#" * 70)

print("\n-- Self-test fungsi normalisasi --")
self_test_normalisasi()

try:
    df_osm = pd.read_csv(OSM_FILE)
    print(f"\nOSM (mentah): {len(df_osm)} baris")
except FileNotFoundError:
    df_osm = pd.DataFrame(columns=["nama", "wifi", "outdoor_seating", "smoking"])
    print("\nOSM tidak ditemukan, dilewati.")

print("\n-- Ekstraksi koordinat --")
coords = df_cafe["link"].apply(extract_lat_lng)
df_cafe["latitude"] = coords.apply(lambda x: x[0])
df_cafe["longitude"] = coords.apply(lambda x: x[1])
print(f"Berhasil ekstrak koordinat: {df_cafe['latitude'].notna().sum()}/{len(df_cafe)}")

df_cafe["nama_normal"] = df_cafe["nama_cafe"].apply(normalize_name)
if "nama" in df_osm.columns:
    df_osm["nama_normal"] = df_osm["nama"].apply(normalize_name)

print("\n-- Deduplikasi cafe --")
before = len(df_cafe)
df_cafe = df_cafe.drop_duplicates(subset="nama_normal", keep="first").reset_index(drop=True)
print(f"Setelah dedup nama identik: {len(df_cafe)} (dari {before})")

to_drop = set()
records = df_cafe[["nama_normal", "latitude", "longitude"]].to_dict("records")
for i in range(len(records)):
    if i in to_drop:
        continue
    for j in range(i + 1, len(records)):
        if j in to_drop:
            continue
        a, b = records[i], records[j]
        if pd.isna(a["latitude"]) or pd.isna(b["latitude"]):
            continue
        jarak = haversine_km(a["latitude"], a["longitude"], b["latitude"], b["longitude"])
        kemiripan = fuzz.ratio(a["nama_normal"], b["nama_normal"])
        if jarak < 0.05 and kemiripan >= DEDUP_NAME_THRESHOLD:
            to_drop.add(j)

if to_drop:
    df_cafe = df_cafe.drop(df_cafe.index[list(to_drop)]).reset_index(drop=True)
print(f"Setelah dedup nama mirip + lokasi berdekatan: {len(df_cafe)} (menghapus {len(to_drop)} tambahan)")

print("\n-- Mencocokkan atribut dari OSM --")
df_cafe["wifi"] = None
df_cafe["outdoor_seating"] = None
df_cafe["smoking"] = None
jumlah_cocok = 0
if "nama_normal" in df_osm.columns and len(df_osm) > 0:
    for idx, row in df_cafe.iterrows():
        best_score, best_match = 0, None
        for _, osm_row in df_osm.iterrows():
            score = fuzz.ratio(row["nama_normal"], osm_row["nama_normal"])
            if score > best_score:
                best_score, best_match = score, osm_row
        if best_score >= OSM_MATCH_THRESHOLD:
            df_cafe.at[idx, "wifi"] = best_match.get("wifi")
            df_cafe.at[idx, "outdoor_seating"] = best_match.get("outdoor_seating")
            df_cafe.at[idx, "smoking"] = best_match.get("smoking")
            jumlah_cocok += 1
print(f"Cafe yang dilengkapi atribut dari OSM: {jumlah_cocok}/{len(df_cafe)}")

print("\n-- Menangani data kosong --")
df_cafe["harga_indikasi"] = df_cafe["harga_indikasi"].fillna("Tidak diketahui")
if "jam_operasional" in df_cafe.columns:
    df_cafe["jam_operasional"] = df_cafe["jam_operasional"].fillna("Tidak diketahui")
if "alamat" in df_cafe.columns:
    df_cafe["alamat"] = df_cafe["alamat"].fillna("Tidak diketahui")
df_cafe["rating_rata2"] = pd.to_numeric(df_cafe["rating_rata2"].astype(str).str.replace(",", "."), errors="coerce")
df_cafe["jumlah_ulasan_total"] = pd.to_numeric(df_cafe["jumlah_ulasan_total"], errors="coerce")

sebelum_buang = len(df_cafe)
df_cafe = df_cafe.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
print(f"Cafe tanpa koordinat valid dibuang: {sebelum_buang - len(df_cafe)}")

print("\n-- Cleaning ulasan --")
before = len(df_reviews)
df_reviews["ulasan"] = df_reviews["ulasan"].astype(str).str.strip()

# Urutan penting: emoji -> spasi, lalu huruf memanjang, lalu normalisasi singkatan
df_reviews["ulasan"] = df_reviews["ulasan"].apply(bersihkan_emoji)
setelah_emoji_kosong = (df_reviews["ulasan"].str.len() == 0).sum()
print(f"Ulasan yang jadi kosong setelah emoji dihapus (cuma berisi emoji): {setelah_emoji_kosong}")

# Huruf memanjang ("enakkk" -> "enak"), memakai kosakata korpus sendiri sebagai pembanding
vocab_korpus = bangun_vocab(df_reviews["ulasan"])
df_reviews["ulasan"] = df_reviews["ulasan"].apply(lambda t: perbaiki_pemanjangan(t, vocab_korpus))
df_reviews["ulasan"] = df_reviews["ulasan"].apply(normalisasi_singkatan)
df_reviews["ulasan"] = df_reviews["ulasan"].str.strip()

df_reviews = df_reviews[df_reviews["ulasan"].str.len() >= 3]
df_reviews = df_reviews.drop_duplicates(subset=["nama_cafe", "ulasan"], keep="first")
df_reviews["rating_bintang"] = pd.to_numeric(df_reviews["rating_bintang"].astype(str).str.replace(",", "."), errors="coerce")
cafe_valid = set(df_cafe["nama_cafe"])
df_reviews = df_reviews[df_reviews["nama_cafe"].isin(cafe_valid)].reset_index(drop=True)
print(f"Ulasan setelah cleaning: {len(df_reviews)} (dari {before})")

# ---- Diagnostik normalisasi ----
sisa = Counter()
token_pendek = Counter()
for u in df_reviews["ulasan"]:
    for t in re.findall(r"[A-Za-z]+", u.lower()):
        if t in KAMUS_NORMALISASI and KAMUS_NORMALISASI[t] != t:
            sisa[t] += 1
        if len(t) <= 4:
            token_pendek[t] += 1

print("\n-- Singkatan dari kamus yang masih tersisa (harusnya kosong) --")
print(sisa.most_common(20) if sisa else "Kosong (bagus)")
print("\n-- 80 token pendek paling sering (cek manual, cari singkatan yang belum ada di kamus) --")
print(token_pendek.most_common(80))

# Diagnostik tambahan: token 5-7 huruf (singkatan yang lebih panjang, mis. "sndiri", "bbrp")
token_sedang = Counter()
for u in df_reviews["ulasan"]:
    for t in re.findall(r"[A-Za-z]+", u.lower()):
        if 5 <= len(t) <= 7:
            token_sedang[t] += 1
print("\n-- 150 token 5-7 huruf paling sering (cek manual) --")
print(token_sedang.most_common(150))

# Diagnostik tambahan: pola aneh (huruf menempel angka, huruf berulang 3x atau lebih)
aneh = Counter()
for u in df_reviews["ulasan"]:
    for m in re.finditer(r"\b[A-Za-z]+\d\b|\b\w*(\w)\1{2,}\w*\b", u.lower()):
        aneh[m.group(0)] += 1
print("\n-- 30 pola aneh paling sering (angka menempel / huruf berulang) --")
print(aneh.most_common(30))

# Diagnostik tambahan: "nya" yang masih berdiri sendiri (lihat konteksnya)
nya_sisa = sum(1 for u in df_reviews["ulasan"] for _ in re.finditer(r"\bnya\b", u))
print(f"\n-- Kata 'nya' yang masih berdiri sendiri: {nya_sisa} kali. Contoh konteks: --")
contoh_nya = []
for u in df_reviews["ulasan"]:
    m = re.search(r".{0,15}\bnya\b.{0,8}", u)
    if m:
        contoh_nya.append(m.group(0))
    if len(contoh_nya) >= 15:
        break
for c in contoh_nya:
    print("   ", repr(c))

# Diagnostik tambahan: kandidat typo (hanya untuk ditinjau manual, tidak diubah otomatis)
print("\n-- Kandidat typo: kata jarang yang mirip kata yang jauh lebih sering (cek manual) --")
kata_sah = muat_kamus_kata_dasar()
if kata_sah:
    print(f"Kamus kata dasar dimuat ({len(kata_sah)} kata): kata sah tidak akan diusulkan sebagai typo.")
else:
    print("Kamus kata dasar tidak tersedia (pip install PySastrawi untuk menyaring kata sah dari daftar ini).")
kandidat_typo = cari_kandidat_typo(bangun_vocab(df_reviews["ulasan"]), kata_sah=kata_sah)
if kandidat_typo:
    pd.DataFrame(
        kandidat_typo,
        columns=["kata_diduga_typo", "freq_typo", "saran_perbaikan", "freq_saran"],
    ).to_csv(KANDIDAT_TYPO_OUT, index=False, encoding="utf-8-sig")
    for w, c, b, cb in kandidat_typo[:60]:
        print(f"   {w} ({c}x)  ->  {b} ({cb}x)")
    print(f"[i] {len(kandidat_typo)} kandidat disimpan ke {KANDIDAT_TYPO_OUT}.")
    print("    Salin ke KAMUS_TYPO HANYA kata yang memang bukan kata dan perbaikannya tidak ambigu.")
    print("    Jangan salin kata yang punya arti sendiri (mis. walau, tepat, taman), walau mirip kata lain.")
    print("    Cek juga kolom saran: bisa salah walau tokennya memang typo (mis. mayan bukan makan, tapi lumayan).")
else:
    print("Tidak ada kandidat.")

kolom_final_cafe = [c for c in [
    "nama_cafe", "link", "latitude", "longitude", "alamat", "rating_rata2",
    "jumlah_ulasan_total", "harga_indikasi", "jam_operasional",
    "wifi", "outdoor_seating", "smoking",
] if c in df_cafe.columns]

df_cafe[kolom_final_cafe].to_csv(CAFE_CLEAN_OUT, index=False, encoding="utf-8-sig")
df_reviews.to_csv(REVIEWS_CLEAN_OUT, index=False, encoding="utf-8-sig")
print(f"\nCafe bersih   : {len(df_cafe)} -> {CAFE_CLEAN_OUT}")
print(f"Ulasan bersih : {len(df_reviews)} -> {REVIEWS_CLEAN_OUT}")


# ==================== TAHAP 3: VALIDASI ====================

print("\n" + "#" * 70)
print("TAHAP 3: VALIDASI KUALITAS DATA")
print("#" * 70)

df_cafe_v = pd.read_csv(CAFE_CLEAN_OUT)
df_reviews_v = pd.read_csv(REVIEWS_CLEAN_OUT)

print("\n=== 1. JUMLAH DATA ===")
print(f"Total cafe   : {len(df_cafe_v)}")
print(f"Total ulasan : {len(df_reviews_v)}")
print(f"Rata-rata ulasan per cafe : {len(df_reviews_v) / len(df_cafe_v):.1f}")

print("\n=== 2. DATA KOSONG PER KOLOM ===")
print("-- cafe_clean.csv --")
print(df_cafe_v.isna().sum())

print("\n=== 3. DUPLIKAT TERSISA ===")
print(f"Nama cafe masih duplikat: {df_cafe_v['nama_cafe'].duplicated().sum()} (harusnya 0)")

print("\n=== 4. RENTANG NILAI ===")
print(f"Rating: {df_cafe_v['rating_rata2'].min()} - {df_cafe_v['rating_rata2'].max()}")
print(f"Latitude: {df_cafe_v['latitude'].min()} - {df_cafe_v['latitude'].max()}")
print(f"Longitude: {df_cafe_v['longitude'].min()} - {df_cafe_v['longitude'].max()}")

print("\n=== 5. CAKUPAN ULASAN ===")
jumlah_per_cafe = df_reviews_v.groupby("nama_cafe").size().reset_index(name="ulasan_terambil")
cek = df_cafe_v.merge(jumlah_per_cafe, on="nama_cafe", how="left")
cek["ulasan_terambil"] = cek["ulasan_terambil"].fillna(0).astype(int)
cek["persentase_cakupan"] = (cek["ulasan_terambil"] / cek["jumlah_ulasan_total"] * 100).round(1)
print(f"Cafe tanpa ulasan sama sekali: {(cek['ulasan_terambil'] == 0).sum()}")
print(f"Rata-rata cakupan: {cek['persentase_cakupan'].mean():.1f}%")
print("\n5 cafe cakupan terendah:")
print(cek.nsmallest(5, "persentase_cakupan")[["nama_cafe", "jumlah_ulasan_total", "ulasan_terambil", "persentase_cakupan"]].to_string())

cek.to_csv(CEK_CAKUPAN_OUT, index=False, encoding="utf-8-sig")
print(f"\n[i] Detail cakupan disimpan ke {CEK_CAKUPAN_OUT}")

print("\n" + "#" * 70)
print("SEMUA TAHAP SELESAI")
print("#" * 70)

######################################################################
TAHAP 1: MERGE HASIL RETRY KE DATA MENTAH UTAMA
######################################################################
Data cafe mentah (sebelum merge retry)   : 446
Data ulasan mentah (sebelum merge retry) : 72915
Ditemukan hasil retry: 19 cafe, 1539 ulasan
Setelah merge retry -> cafe: 446, ulasan: 72915
(file_cafe_lengkap.csv dan file_google_maps_reviews.csv sudah diperbarui)

######################################################################
TAHAP 2: CLEANING, DEDUPLIKASI, DAN GABUNG ATRIBUT OSM
######################################################################

-- Self-test fungsi normalisasi --
Self-test normalisasi: 22/22 lulus
Self-test emoji: 'enak😍yg bagus' -> 'enak yang bagus' (harusnya 'enak yang bagus')

OSM (mentah): 74 baris

-- Ekstraksi koordinat --
Berhasil ekstrak koordinat: 446/446

-- Deduplikasi cafe --
Setelah dedup nama identik: 440 (dari 446)
Setelah dedup nama mirip + lokasi berdekat

In [10]:
pip install pandas Sastrawi

Note: you may need to restart the kernel to use updated packages.


In [1]:
# =============================================================
# 01_praproses.py
# Tahap 3.5.1 Praproses Data
#   A. Praproses atribut coffee shop (normalisasi kategori harga)
#   B. Praproses teks ulasan (pecah kalimat, case folding,
#      normalisasi, tokenisasi, stopword, stemming)
# Input : cafe_clean.csv, reviews_clean.csv
# Output: cafe_praproses.csv, kalimat_praproses.csv, kamus_stem.csv
# =============================================================
import os
import re
from multiprocessing import Pool, cpu_count

import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# -------------------------------------------------------------
# A. FUNGSI PRAPROSES ATRIBUT COFFEE SHOP
# -------------------------------------------------------------
def batas_atas_harga(teks):
    """Ambil batas atas rentang harga, contoh 'Rp 25–50 rb' -> 50000."""
    m = re.match(r"^\s*Rp\s*([\d.]+)\s*[–-]\s*([\d.]+)\s*(rb)?\s*$", str(teks))
    if not m:
        return None  # 'Tidak diketahui' atau isi kolom yang tidak valid
    nilai = int(m.group(2).replace(".", ""))
    return nilai * 1000 if m.group(3) else nilai

def kategori_harga(batas):
    if batas is None or pd.isna(batas):
        return "tidak_diketahui"
    if batas <= 25000:
        return "murah"
    if batas <= 50000:
        return "sedang"
    return "mahal"

# -------------------------------------------------------------
# B. FUNGSI PRAPROSES TEKS ULASAN
# -------------------------------------------------------------
# B1. Pemecahan kalimat: tanda baca akhir kalimat dan kata penghubung
#     pertentangan, supaya "suasana nyaman, sayangnya pelayanan lambat"
#     menjadi dua bagian yang dapat dianalisis per aspek.
PEMISAH_KALIMAT = re.compile(r"[.!?…]+\s*|\s*\n+\s*")
PENGHUBUNG = re.compile(r"\s*,?\s+\b(tapi|tetapi|namun|sayangnya|sayang nya|hanya saja)\b", re.IGNORECASE)

def pecah_kalimat(teks):
    hasil = []
    for bagian in PEMISAH_KALIMAT.split(str(teks)):
        bagian = PENGHUBUNG.sub(lambda m: " ||| " + m.group(1), bagian)
        hasil += [b.strip(" ,;:-") for b in bagian.split("|||")]
    return [h for h in hasil if h]

# B2. Normalisasi kata tidak baku (fokus pada kata yang memengaruhi
#     makna sentimen dan nama aspek)
KAMUS_NORMALISASI = {
    # negasi
    "gak": "tidak", "ga": "tidak", "gk": "tidak", "nggak": "tidak", "ngga": "tidak",
    "enggak": "tidak", "engga": "tidak", "tdk": "tidak", "kagak": "tidak",
    "blm": "belum", "blum": "belum", "krg": "kurang", "kurg": "kurang",
    # kata umum
    "yg": "yang", "dgn": "dengan", "dg": "dengan", "utk": "untuk", "untk": "untuk",
    "krn": "karena", "karna": "karena", "tp": "tapi", "tpi": "tapi", "jg": "juga",
    "sy": "saya", "aja": "saja", "aj": "saja", "udah": "sudah", "udh": "sudah", "sdh": "sudah",
    "bgt": "banget", "bngt": "banget", "bener": "benar", "bnr": "benar", "trs": "terus",
    "tmpt": "tempat", "tmpat": "tempat", "pelayananya": "pelayanan", "pelayanannya": "pelayanan",
    "harganya": "harga", "suasananya": "suasana", "tempatnya": "tempat",
    "mantul": "mantap", "mantab": "mantap", "rekomen": "rekomendasi", "recomend": "rekomendasi",
    "recommended": "rekomendasi", "rekomended": "rekomendasi",
    # aspek wifi
    "wifinya": "wifi", "wify": "wifi", "wf": "wifi", "inet": "internet",
    "sinyalnya": "sinyal", "koneksinya": "koneksi",
}

def bersihkan(teks):
    teks = teks.lower()                                   # case folding
    teks = re.sub(r"\bwi[\s-]fi\b", "wifi", teks)         # "wi fi" / "wi-fi"
    teks = re.sub(r"\bworth\s+it\b", "worth_it", teks)    # frasa indikator aspek harga
    teks = re.sub(r"[^a-z_\s]", " ", teks)                # buang angka, emoji, tanda baca
    teks = re.sub(r"(.)\1{2,}", r"\1", teks)              # "ngobroll" -> "ngobrol"
    return re.sub(r"\s+", " ", teks).strip()

def tokenisasi(teks):
    return [KAMUS_NORMALISASI.get(t, t) for t in teks.split()]

# B3. Stopword: daftar Sastrawi, tetapi kata negasi dan pertentangan
#     DIPERTAHANKAN karena membalik makna sentimen ("tidak nyaman").
PENGECUALIAN = {"tidak", "belum", "bukan", "tanpa", "jangan", "kurang", "tapi", "namun", "agak"}
STOPWORD = set(StopWordRemoverFactory().get_stop_words()) - PENGECUALIAN

# B4. Stemming per kata unik, paralel, dan disimpan ke kamus_stem.csv
#     supaya proses yang lama ini cukup dijalankan sekali.
_stemmer = None
def _stem_satu(kata):
    global _stemmer
    if _stemmer is None:
        _stemmer = StemmerFactory().create_stemmer()
    return kata, _stemmer.stem(kata)

def buat_kamus_stem(kosakata, file="kamus_stem.csv"):
    kamus = {}
    if os.path.exists(file):
        lama = pd.read_csv(file, keep_default_na=False)
        kamus = dict(zip(lama["kata"], lama["kata_dasar"]))
    akar = set(StemmerFactory().get_words())
    for k in kosakata:
        # kata dasar, kata pendek, dan frasa gabungan tidak perlu di-stem
        if k in akar or len(k) <= 3 or "_" in k:
            kamus.setdefault(k, k)
    sisa = sorted(k for k in kosakata if k not in kamus)
    if sisa:
        print(f"Stemming {len(sisa)} kata unik dengan {cpu_count()} proses ...")
        with Pool(cpu_count()) as pool:
            for i, (k, s) in enumerate(pool.imap_unordered(_stem_satu, sisa, chunksize=200), 1):
                kamus[k] = s
                if i % 2000 == 0:
                    print(f"  {i}/{len(sisa)}")
        pd.DataFrame({"kata": list(kamus), "kata_dasar": list(kamus.values())}).to_csv(file, index=False)
    return kamus

# -------------------------------------------------------------
# PROSES UTAMA
# -------------------------------------------------------------
def main():
    cafe = pd.read_csv("cafe_clean.csv").reset_index(drop=True)
    ulasan = pd.read_csv("reviews_clean.csv").reset_index(drop=True)

    # A. atribut coffee shop
    cafe.insert(0, "id_cafe", cafe.index + 1)
    cafe["harga_batas_atas"] = cafe["harga_indikasi"].apply(batas_atas_harga).astype("Int64")
    cafe["kategori_harga"] = cafe["harga_batas_atas"].apply(kategori_harga)
    jml = ulasan.groupby("nama_cafe").size().rename("jumlah_ulasan_terambil")
    cafe = cafe.merge(jml, on="nama_cafe", how="left")
    cafe["jumlah_ulasan_terambil"] = cafe["jumlah_ulasan_terambil"].fillna(0).astype(int)

    # B. teks ulasan
    ulasan.insert(0, "id_ulasan", ulasan.index + 1)
    ulasan = ulasan.merge(cafe[["id_cafe", "nama_cafe"]], on="nama_cafe", how="left")

    baris = []
    for row in ulasan.itertuples(index=False):
        for i, kal in enumerate(pecah_kalimat(row.ulasan), start=1):
            token = tokenisasi(bersihkan(kal))
            token_sw = [t for t in token if t not in STOPWORD]
            if not token_sw:
                continue
            baris.append({
                "id_ulasan": row.id_ulasan, "id_cafe": row.id_cafe, "nama_cafe": row.nama_cafe,
                "rating_bintang": row.rating_bintang, "urutan_kalimat": i,
                "kalimat_asli": kal,
                "kalimat_normal": " ".join(token),          # untuk deteksi aspek (tahap 2)
                "token_tanpa_stopword": " ".join(token_sw),
            })
    kalimat = pd.DataFrame(baris)

    kosakata = set(" ".join(kalimat["token_tanpa_stopword"]).split())
    kamus = buat_kamus_stem(kosakata)
    kalimat["token_stem"] = kalimat["token_tanpa_stopword"].apply(
        lambda s: " ".join(kamus.get(t, t) for t in s.split()))  # untuk TF-IDF dan SVM
    kalimat.insert(0, "id_kalimat", range(1, len(kalimat) + 1))

    cafe.to_csv("cafe_praproses.csv", index=False)
    kalimat.to_csv("kalimat_praproses.csv", index=False)

    # RINGKASAN (bahan BAB 4.2)
    print("\n=== Praproses atribut coffee shop ===")
    print("Jumlah coffee shop :", len(cafe))
    print(cafe["kategori_harga"].value_counts().to_string())
    tidak_valid = cafe["harga_batas_atas"].isna() & (cafe["harga_indikasi"] != "Tidak diketahui")
    print("Isi harga tidak valid:", tidak_valid.sum(), list(cafe.loc[tidak_valid, "nama_cafe"]))
    print("Coffee shop tanpa ulasan:", (cafe["jumlah_ulasan_terambil"] == 0).sum())
    print("\n=== Praproses teks ulasan ===")
    print("Jumlah ulasan           :", len(ulasan))
    print("Ulasan terpotong (…)    :", ulasan["ulasan"].astype(str).str.strip().str.endswith("…").sum())
    print("Jumlah kalimat          :", len(kalimat))
    print("Rata-rata kalimat/ulasan:", round(len(kalimat) / len(ulasan), 2))
    print("Kata unik sebelum stem  :", len(kosakata))
    print("Kata unik setelah stem  :", len({kamus.get(k, k) for k in kosakata}))
    print("\nContoh hasil:")
    print(kalimat[["kalimat_asli", "kalimat_normal", "token_stem"]].head(5).to_string())

if __name__ == "__main__":
    main()


=== Praproses atribut coffee shop ===
Jumlah coffee shop : 440
kategori_harga
sedang             212
murah               99
tidak_diketahui     89
mahal               40
Isi harga tidak valid: 1 ['Naraya Coffee Shop & Beyond']
Coffee shop tanpa ulasan: 14

=== Praproses teks ulasan ===
Jumlah ulasan           : 71483
Ulasan terpotong (…)    : 18469
Jumlah kalimat          : 136482
Rata-rata kalimat/ulasan: 1.91
Kata unik sebelum stem  : 30022
Kata unik setelah stem  : 24111

Contoh hasil:
                                                                                                         kalimat_asli                                                                                                    kalimat_normal                                                                                      token_stem
0  Eastman Coffee House punya daya tarik yang cukup kuat sebagai tempat nongkrong yang “niat”—bukan sekadar jual kopi  eastman coffee house punya daya tarik yang cukup kuat seba

In [2]:
pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [3]:
# =============================================================
# 02_ekstraksi_aspek.py
# Tahap 3.5.2 (bagian 1) Analisis Sentimen Berbasis Aspek
#   A. Ekstraksi aspek berbasis kata kunci indikator (Tabel 3.1)
#   B. Pengambilan sampel kalimat untuk pelabelan manual
# Input : kalimat_praproses.csv
# Output: aspek_kalimat.csv, sampel_label.xlsx
# =============================================================
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation

# -------------------------------------------------------------
# A. KAMUS KATA KUNCI INDIKATOR ASPEK
#    (perluasan dari contoh pada Tabel 3.1, dicocokkan pada
#     kolom kalimat_normal yang belum di-stem)
# -------------------------------------------------------------
KATA_KUNCI = {
    "wifi": {"wifi", "internet", "sinyal", "koneksi", "jaringan", "hotspot"},
    "harga": {"harga", "murah", "mahal", "terjangkau", "worth_it", "pricey", "overprice",
              "overpriced", "affordable", "kantong", "ekonomis", "murmer", "expensive"},
    "suasana": {"suasana", "nyaman", "estetik", "aesthetic", "estetis", "cozy", "berisik",
                "bising", "ramai", "sepi", "tenang", "adem", "sejuk", "gerah", "pengap",
                "interior", "desain", "dekorasi", "vibes", "vibe", "ambience", "ambiance",
                "instagramable", "homey"},
    "pelayanan": {"pelayanan", "layanan", "pelayan", "service", "servis", "staff", "staf",
                  "karyawan", "barista", "waiter", "waitress", "kasir", "pegawai", "ramah",
                  "jutek", "judes", "sopan", "responsif", "sigap", "cekatan", "informatif",
                  "helpful", "friendly"},
}
# kata yang membuat "ramah"/"friendly" tidak bermakna pelayanan:
# "ramah di kantong", "budget friendly" (harga), "ramah anak", "cashless friendly" (fasilitas)
BUKAN_PELAYANAN = {"kantong", "budget", "pocket", "dompet", "anak", "kids", "pet",
                   "family", "keluarga", "cashless", "lingkungan"}

def deteksi_aspek(kalimat_normal):
    token = set(str(kalimat_normal).split())
    hasil = {}
    for aspek, kunci in KATA_KUNCI.items():
        cocok = token & kunci
        if aspek == "pelayanan" and token & BUKAN_PELAYANAN:
            cocok = cocok - {"ramah", "friendly"}
        if cocok:
            hasil[aspek] = ", ".join(sorted(cocok))
    return hasil

def main():
    kalimat = pd.read_csv("kalimat_praproses.csv", keep_default_na=False)

    baris = []
    for row in kalimat.itertuples(index=False):
        for aspek, kunci in deteksi_aspek(row.kalimat_normal).items():
            baris.append({
                "id_kalimat": row.id_kalimat, "id_ulasan": row.id_ulasan, "id_cafe": row.id_cafe,
                "nama_cafe": row.nama_cafe, "rating_bintang": row.rating_bintang,
                "aspek": aspek, "kata_kunci": kunci,
                "kalimat_asli": row.kalimat_asli, "kalimat_normal": row.kalimat_normal,
                "token_stem": row.token_stem,
            })
    aspek_kalimat = pd.DataFrame(baris)
    aspek_kalimat.insert(0, "id_aspek_kalimat", range(1, len(aspek_kalimat) + 1))
    aspek_kalimat.to_csv("aspek_kalimat.csv", index=False)

    # ---------------------------------------------------------
    # B. SAMPEL UNTUK PELABELAN MANUAL
    #    250 kalimat per aspek. Kalimat dari ulasan berbintang 1-3
    #    diambil lebih banyak (40%) agar sentimen negatif cukup
    #    terwakili, karena 75% ulasan berbintang 5.
    # ---------------------------------------------------------
    N_PER_ASPEK, PORSI_RENDAH, SEED = 250, 0.4, 42
    sampel = []
    for aspek, grup in aspek_kalimat.groupby("aspek"):
        rendah = grup[grup["rating_bintang"] <= 3]
        tinggi = grup[grup["rating_bintang"] >= 4]
        n_rendah = min(len(rendah), int(N_PER_ASPEK * PORSI_RENDAH))
        n_tinggi = min(len(tinggi), N_PER_ASPEK - n_rendah)
        sampel.append(rendah.sample(n_rendah, random_state=SEED))
        sampel.append(tinggi.sample(n_tinggi, random_state=SEED))
    sampel = pd.concat(sampel).sample(frac=1, random_state=SEED)  # acak urutan
    sampel = sampel[["id_aspek_kalimat", "id_kalimat", "aspek", "kata_kunci", "kalimat_asli"]]

    wb = Workbook()
    ws = wb.active
    ws.title = "label"
    ws.append(list(sampel.columns) + ["label"])
    for r in sampel.itertuples(index=False):
        ws.append(list(r) + [""])
    for c in ws[1]:
        c.font = Font(bold=True)
        c.fill = PatternFill("solid", fgColor="D9D9D9")
    for col, w in {"A": 16, "B": 11, "C": 11, "D": 20, "E": 90, "F": 12}.items():
        ws.column_dimensions[col].width = w
    for row in ws.iter_rows(min_row=2, min_col=5, max_col=5):
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
    dv = DataValidation(type="list", formula1='"positif,negatif,netral"', allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(f"F2:F{ws.max_row}")
    ws.freeze_panes = "A2"

    panduan = wb.create_sheet("panduan")
    for baris_p in [
        ["PANDUAN PELABELAN"],
        ["Beri label HANYA berdasarkan opini terhadap aspek pada kolom 'aspek', bukan kesan ulasan secara keseluruhan."],
        ["positif", "Opini baik terhadap aspek. Contoh: 'wifinya kencang', 'harganya terjangkau', 'tempatnya nyaman', 'baristanya ramah'."],
        ["negatif", "Opini buruk terhadap aspek. Contoh: 'wifi sering putus', 'harga agak mahal', 'terlalu berisik', 'kasirnya jutek'."],
        ["netral", "Aspek disebut tanpa opini, atau opininya tidak jelas. Contoh: 'ada wifi', 'harga mulai 20 ribu', 'pelayanan standar'."],
        ["Catatan", "Kalau kata kunci terdeteksi tetapi kalimat sebenarnya tidak membahas aspek itu (misal 'nyaman' untuk rasa minuman), beri label netral."],
    ]:
        panduan.append(baris_p)
    panduan["A1"].font = Font(bold=True)
    panduan.column_dimensions["A"].width = 12
    panduan.column_dimensions["B"].width = 110
    wb.save("sampel_label.xlsx")

    # ---------------------------------------------------------
    # RINGKASAN (bahan BAB 4.3.1)
    # ---------------------------------------------------------
    total = len(kalimat)
    beraspek = aspek_kalimat["id_kalimat"].nunique()
    print("=== Ekstraksi aspek ===")
    print("Jumlah kalimat               :", total)
    print("Kalimat yang memuat aspek    :", beraspek, f"({beraspek / total:.1%})")
    print("Kalimat tanpa aspek (dibuang):", total - beraspek)
    print("Pasangan kalimat-aspek       :", len(aspek_kalimat))
    print("\nJumlah kalimat per aspek:")
    print(aspek_kalimat["aspek"].value_counts().to_string())
    print("\nJumlah coffee shop yang aspeknya dibahas minimal 3 kalimat:")
    per_cafe = aspek_kalimat.groupby(["aspek", "id_cafe"]).size()
    print((per_cafe >= 3).groupby("aspek").sum().to_string())
    print("\n=== Sampel pelabelan ===")
    print(sampel.groupby("aspek").size().to_string())
    print("Total sampel:", len(sampel))

if __name__ == "__main__":
    main()

=== Ekstraksi aspek ===
Jumlah kalimat               : 136482
Kalimat yang memuat aspek    : 49848 (36.5%)
Kalimat tanpa aspek (dibuang): 86634
Pasangan kalimat-aspek       : 58643

Jumlah kalimat per aspek:
aspek
suasana      28572
pelayanan    16423
harga        12072
wifi          1576

Jumlah coffee shop yang aspeknya dibahas minimal 3 kalimat:
aspek
harga        329
pelayanan    337
suasana      341
wifi         162

=== Sampel pelabelan ===
aspek
harga        250
pelayanan    250
suasana      250
wifi         250
Total sampel: 1000


In [4]:
pip install pandas scikit-learn openpyxl joblib

Note: you may need to restart the kernel to use updated packages.


In [5]:
# =============================================================
# 03_klasifikasi_sentimen.py
# Tahap 3.5.2 (bagian 2) Analisis Sentimen Berbasis Aspek
#   A. Pembagian data latih dan data uji (80:20)
#   B. TF-IDF + Support Vector Machine (pemilihan kernel dan C
#      dengan 5-fold cross validation pada data latih)
#   C. Evaluasi pada data uji (bahan BAB 4.7.3)
#   D. Klasifikasi seluruh pasangan kalimat-aspek
#   E. Agregasi menjadi badge per coffee shop per aspek
# Input : aspek_kalimat.csv, sampel_label.xlsx (sudah dilabeli)
# Output: sentimen_kalimat.csv, badge_cafe.csv, evaluasi_svm.txt,
#         confusion_matrix.csv, model_svm.joblib
# =============================================================
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

SEED = 42
LABEL = ["positif", "negatif", "netral"]
MIN_KALIMAT_BADGE = 3
TEKS_BADGE = {
    "wifi": ("Wifi Kencang", "Wifi Kurang Stabil"),
    "harga": ("Harga Terjangkau", "Harga Kurang Bersahabat"),
    "suasana": ("Suasana Nyaman", "Kurang Cocok untuk Fokus"),
    "pelayanan": ("Pelayanan Ramah", "Pelayanan Perlu Ditingkatkan"),
}

def teks_model(df):
    # nama aspek disisipkan sebagai token agar model tahu aspek yang dinilai
    return "aspek_" + df["aspek"] + " " + df["token_stem"].fillna("")

def tentukan_badge(row):
    pos, neg, net = row["n_positif"], row["n_negatif"], row["n_netral"]
    if row["n_total"] < MIN_KALIMAT_BADGE or (net > pos and net > neg) or pos == neg:
        return pd.Series({"sentimen_badge": "", "badge": ""})
    if pos > neg:
        return pd.Series({"sentimen_badge": "positif", "badge": TEKS_BADGE[row["aspek"]][0]})
    return pd.Series({"sentimen_badge": "negatif", "badge": TEKS_BADGE[row["aspek"]][1]})

def main():
    data = pd.read_csv("aspek_kalimat.csv", keep_default_na=False)
    label = pd.read_excel("sampel_label.xlsx", sheet_name="label")[["id_aspek_kalimat", "label"]]
    label["label"] = label["label"].astype(str).str.strip().str.lower()
    kosong = (~label["label"].isin(LABEL)).sum()
    label = label[label["label"].isin(LABEL)]
    berlabel = data.merge(label, on="id_aspek_kalimat")

    # A. Data latih dan data uji 80:20 (stratifikasi aspek + label)
    strata = berlabel["aspek"] + "_" + berlabel["label"]
    latih, uji = train_test_split(berlabel, test_size=0.2, stratify=strata, random_state=SEED)

    # B. TF-IDF + SVM, kernel linear / polynomial / RBF dibandingkan
    #    dengan 5-fold cross validation pada data latih saja
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
        ("svm", SVC(class_weight="balanced", random_state=SEED)),
    ])
    grid = GridSearchCV(
        pipeline,
        {"svm__kernel": ["linear", "poly", "rbf"], "svm__C": [0.1, 1, 10]},
        cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
        scoring="f1_macro", n_jobs=-1,
    )
    grid.fit(teks_model(latih), latih["label"])
    model = grid.best_estimator_

    # C. Evaluasi pada data uji
    prediksi_uji = model.predict(teks_model(uji))
    akurasi = accuracy_score(uji["label"], prediksi_uji)
    laporan = classification_report(uji["label"], prediksi_uji, labels=LABEL, digits=4, zero_division=0)
    cm = pd.DataFrame(confusion_matrix(uji["label"], prediksi_uji, labels=LABEL),
                      index=[f"aktual_{l}" for l in LABEL], columns=[f"prediksi_{l}" for l in LABEL])
    hasil_cv = pd.DataFrame(grid.cv_results_)[["param_svm__kernel", "param_svm__C", "mean_test_score"]]
    hasil_cv = hasil_cv.sort_values("mean_test_score", ascending=False)
    uji = uji.assign(prediksi=prediksi_uji)
    per_aspek = uji.groupby("aspek").apply(lambda g: accuracy_score(g["label"], g["prediksi"]), include_groups=False)

    teks = []
    teks.append(f"Data berlabel dipakai : {len(berlabel)} (label kosong dilewati: {kosong})")
    teks.append(f"Data latih / data uji : {len(latih)} / {len(uji)}")
    teks.append("\nDistribusi label data berlabel:\n" + pd.crosstab(berlabel["aspek"], berlabel["label"]).to_string())
    teks.append("\nHasil 5-fold cross validation (F1 macro) pada data latih:\n" + hasil_cv.to_string(index=False))
    teks.append(f"\nParameter terbaik     : {grid.best_params_}")
    teks.append(f"\nAkurasi data uji      : {akurasi:.4f}")
    teks.append("\nClassification report data uji:\n" + laporan)
    teks.append("Confusion matrix data uji:\n" + cm.to_string())
    teks.append("\nAkurasi data uji per aspek:\n" + per_aspek.round(4).to_string())
    with open("evaluasi_svm.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(teks))
    cm.to_csv("confusion_matrix.csv")
    joblib.dump(model, "model_svm.joblib")
    print("\n".join(teks))

    # D. Klasifikasi seluruh pasangan kalimat-aspek.
    #    Kalimat yang sudah dilabeli manual memakai label manual.
    data["sentimen"] = model.predict(teks_model(data))
    data["sumber_label"] = "model"
    peta_manual = dict(zip(berlabel["id_aspek_kalimat"], berlabel["label"]))
    manual = data["id_aspek_kalimat"].isin(peta_manual)
    data.loc[manual, "sentimen"] = data.loc[manual, "id_aspek_kalimat"].map(peta_manual)
    data.loc[manual, "sumber_label"] = "manual"
    data.to_csv("sentimen_kalimat.csv", index=False)

    # E. Agregasi badge per coffee shop per aspek
    hitung = (data.groupby(["id_cafe", "nama_cafe", "aspek", "sentimen"]).size()
                  .unstack(fill_value=0).reindex(columns=LABEL, fill_value=0)
                  .rename(columns=lambda c: f"n_{c}").reset_index())
    hitung["n_total"] = hitung[["n_positif", "n_negatif", "n_netral"]].sum(axis=1)
    hitung["persen_positif"] = (100 * hitung["n_positif"] / hitung["n_total"]).round(1)
    hitung[["sentimen_badge", "badge"]] = hitung.apply(tentukan_badge, axis=1)
    hitung.to_csv("badge_cafe.csv", index=False)

    print("\n=== Hasil klasifikasi seluruh data ===")
    print(pd.crosstab(data["aspek"], data["sentimen"]).to_string())
    print("\n=== Jumlah coffee shop per badge ===")
    print(hitung[hitung["badge"] != ""].groupby(["aspek", "badge"]).size().to_string())
    print("\nCoffee shop yang memiliki minimal satu badge:",
          hitung.loc[hitung["badge"] != "", "id_cafe"].nunique())

if __name__ == "__main__":
    main()

Data berlabel dipakai : 1000 (label kosong dilewati: 0)
Data latih / data uji : 800 / 200

Distribusi label data berlabel:
label      negatif  netral  positif
aspek                              
harga           76      12      162
pelayanan       83      18      149
suasana         35      23      192
wifi           102      48      100

Hasil 5-fold cross validation (F1 macro) pada data latih:
param_svm__kernel  param_svm__C  mean_test_score
           linear           1.0         0.672377
           linear          10.0         0.668427
              rbf          10.0         0.638784
              rbf           1.0         0.636164
           linear           0.1         0.508154
             poly           1.0         0.440740
             poly          10.0         0.352976
              rbf           0.1         0.222843
             poly           0.1         0.170088

Parameter terbaik     : {'svm__C': 1, 'svm__kernel': 'linear'}

Akurasi data uji      : 0.8600

Classification 

In [6]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [7]:
# =============================================================
# 04_dokumen_fitur_cbf.py
# Tahap 3.5.3 Pembentukan Dokumen Fitur dan Content-Based Filtering
#   A. Dokumen fitur per coffee shop = kategori harga + label aspek
#   B. Pembobotan TF-IDF (TF = proporsi kalimat yang mendukung label)
#   C. Normalisasi vektor (panjang = 1) agar cosine similarity di PHP
#      cukup berupa perkalian titik
#   D. Contoh perhitungan cosine similarity (bahan BAB 4.4.3)
# Input : cafe_praproses.csv, badge_cafe.csv
# Output: dokumen_fitur.csv, idf_fitur.csv, bobot_tfidf.csv
# =============================================================
import math
import pandas as pd

def main():
    cafe = pd.read_csv("cafe_praproses.csv")
    badge = pd.read_csv("badge_cafe.csv", keep_default_na=False)

    # ---------------------------------------------------------
    # A. Dokumen fitur
    #    - budget_murah / budget_sedang / budget_mahal  (TF = 1)
    #    - <aspek>_positif atau <aspek>_negatif dari badge
    #      (TF = proporsi kalimat bersentimen sesuai badge)
    # ---------------------------------------------------------
    baris = []
    for c in cafe.itertuples(index=False):
        if c.kategori_harga != "tidak_diketahui":
            baris.append((c.id_cafe, f"budget_{c.kategori_harga}", 1.0))
    for b in badge[badge["badge"] != ""].itertuples(index=False):
        n_sesuai = b.n_positif if b.sentimen_badge == "positif" else b.n_negatif
        baris.append((b.id_cafe, f"{b.aspek}_{b.sentimen_badge}", round(n_sesuai / b.n_total, 4)))
    tf = pd.DataFrame(baris, columns=["id_cafe", "fitur", "tf"])

    dokumen = (tf.groupby("id_cafe")["fitur"].apply(lambda s: " ".join(sorted(s)))
                 .reindex(cafe["id_cafe"], fill_value="").rename("dokumen").reset_index())
    dokumen = cafe[["id_cafe", "nama_cafe"]].merge(dokumen, on="id_cafe")
    dokumen.to_csv("dokumen_fitur.csv", index=False)

    # ---------------------------------------------------------
    # B. IDF = ln(N / df) + 1  (N = jumlah coffee shop)
    # ---------------------------------------------------------
    N = len(cafe)
    idf = tf.groupby("fitur")["id_cafe"].nunique().rename("df").reset_index()
    idf["idf"] = idf["df"].apply(lambda df: round(math.log(N / df) + 1, 6))
    idf.to_csv("idf_fitur.csv", index=False)

    bobot = tf.merge(idf[["fitur", "idf"]], on="fitur")
    bobot["bobot"] = bobot["tf"] * bobot["idf"]

    # ---------------------------------------------------------
    # C. Normalisasi L2 per coffee shop
    # ---------------------------------------------------------
    norma = bobot.groupby("id_cafe")["bobot"].apply(lambda w: math.sqrt((w ** 2).sum())).rename("norma")
    bobot = bobot.merge(norma, on="id_cafe")
    bobot["bobot_normal"] = (bobot["bobot"] / bobot["norma"]).round(6)
    bobot = bobot.drop(columns="norma").sort_values(["id_cafe", "fitur"])
    bobot.to_csv("bobot_tfidf.csv", index=False)

    # ---------------------------------------------------------
    # D. Contoh cosine similarity (logika yang sama nanti di PHP)
    #    TF query: fitur yang dipilih pengguna = 2, aspek positif
    #    lain = 1, karena pengguna tetap menginginkan coffee shop
    #    yang baik di semua aspek, hanya prioritasnya berbeda.
    #    Contoh: budget murah, prioritas wifi kencang dan suasana nyaman
    # ---------------------------------------------------------
    ASPEK_POSITIF = ["wifi_positif", "harga_positif", "suasana_positif", "pelayanan_positif"]
    dipilih = ["budget_murah", "wifi_positif", "suasana_positif"]
    tf_query = {f: 1.0 for f in ASPEK_POSITIF}
    tf_query.update({f: 2.0 for f in dipilih})
    peta_idf = dict(zip(idf["fitur"], idf["idf"]))
    q = {f: t * peta_idf.get(f, 0.0) for f, t in tf_query.items()}
    norma_q = math.sqrt(sum(v ** 2 for v in q.values()))
    q = {f: v / norma_q for f, v in q.items()}

    skor = (bobot[bobot["fitur"].isin(q)]
            .assign(kontribusi=lambda d: d["bobot_normal"] * d["fitur"].map(q))
            .groupby("id_cafe")["kontribusi"].sum().rename("cosine_similarity").reset_index())
    skor = skor.merge(dokumen, on="id_cafe").sort_values("cosine_similarity", ascending=False)

    print("=== Dokumen fitur ===")
    print("Jumlah coffee shop            :", N)
    print("Dokumen fitur tidak kosong    :", (dokumen["dokumen"] != "").sum())
    print("Dokumen fitur kosong          :", (dokumen["dokumen"] == "").sum())
    print("\n=== IDF setiap label fitur ===")
    print(idf.sort_values("idf").to_string(index=False))
    print("\n=== Contoh cosine similarity ===")
    print("Fitur dipilih (TF=2):", ", ".join(dipilih), "| aspek positif lain TF=1")
    print(skor.head(10)[["nama_cafe", "dokumen", "cosine_similarity"]].round(4).to_string(index=False))

if __name__ == "__main__":
    main()

=== Dokumen fitur ===
Jumlah coffee shop            : 440
Dokumen fitur tidak kosong    : 384
Dokumen fitur kosong          : 56

=== IDF setiap label fitur ===
            fitur  df      idf
  suasana_positif 340 1.257829
pelayanan_positif 324 1.306031
    harga_positif 320 1.318454
    budget_sedang 212 1.730188
     wifi_positif 110 2.386294
     budget_murah  99 2.491655
     budget_mahal  40 3.397895
     wifi_negatif  25 3.867899
pelayanan_negatif  11 4.688879
    harga_negatif   6 5.295015

=== Contoh cosine similarity ===
Fitur dipilih (TF=2): budget_murah, wifi_positif, suasana_positif | aspek positif lain TF=1
                nama_cafe                                                                   dokumen  cosine_similarity
     Bento Kopi Tembalang budget_murah harga_positif pelayanan_positif suasana_positif wifi_positif             0.9846
                BURJO BAE budget_murah harga_positif pelayanan_positif suasana_positif wifi_positif             0.9830
Warlook: Warkop

In [9]:
# =============================================================
# 05_data_final.py
# Tahap 3.5.4 Penyusunan Data Final
#   Menggabungkan seluruh hasil olah data menjadi 6 tabel:
#   cafe, ulasan, kalimat_aspek, badge_aspek, fitur_idf, bobot_tfidf
# Input : cafe_praproses.csv, reviews_clean.csv, sentimen_kalimat.csv,
#         badge_cafe.csv, dokumen_fitur.csv, idf_fitur.csv, bobot_tfidf.csv
# Output: folder data_final/ (6 file CSV) dan db_coffee_shop.sql
# Kebutuhan: pip install pandas
# =============================================================
import math
import os
import re
import pandas as pd

NAMA_DB = "db_coffee_shop"
FOLDER = "data_final"

def bersihkan_jam(teks):
    # "Senin,09.00 hingga 22.00, Salin jam buka" -> "09.00 - 22.00"
    t = re.sub(r"^\s*Senin\s*,\s*", "", str(teks))
    t = re.sub(r",?\s*Salin jam buka\s*$", "", t)
    return t.replace(" hingga ", " - ").strip()

def siapkan_tabel():
    cafe = pd.read_csv("cafe_praproses.csv")
    dokumen = pd.read_csv("dokumen_fitur.csv", keep_default_na=False)
    cafe = cafe.merge(dokumen[["id_cafe", "dokumen"]], on="id_cafe", how="left")
    # isi harga yang tidak valid (bukan rentang harga) diganti "Tidak diketahui"
    tidak_valid = cafe["harga_batas_atas"].isna()
    cafe.loc[tidak_valid, "harga_indikasi"] = "Tidak diketahui"
    cafe["jam_operasional"] = cafe["jam_operasional"].apply(bersihkan_jam)
    cafe = cafe.rename(columns={"dokumen": "dokumen_fitur"})[[
        "id_cafe", "nama_cafe", "link", "latitude", "longitude", "alamat", "rating_rata2",
        "jumlah_ulasan_total", "jumlah_ulasan_terambil", "harga_indikasi", "kategori_harga",
        "jam_operasional", "dokumen_fitur"]]

    ulasan = pd.read_csv("reviews_clean.csv").reset_index(drop=True)
    ulasan.insert(0, "id_ulasan", ulasan.index + 1)          # sama dengan tahap praproses
    ulasan = ulasan.merge(cafe[["id_cafe", "nama_cafe"]], on="nama_cafe")
    ulasan = ulasan[["id_ulasan", "id_cafe", "ulasan", "rating_bintang"]].sort_values("id_ulasan")

    kalimat = pd.read_csv("sentimen_kalimat.csv", keep_default_na=False)[[
        "id_aspek_kalimat", "id_kalimat", "id_ulasan", "id_cafe", "aspek", "sentimen",
        "sumber_label", "kalimat_asli"]]

    badge = pd.read_csv("badge_cafe.csv", keep_default_na=False).drop(columns="nama_cafe")
    idf = pd.read_csv("idf_fitur.csv")
    bobot = pd.read_csv("bobot_tfidf.csv")
    return {"cafe": cafe, "ulasan": ulasan, "kalimat_aspek": kalimat,
            "badge_aspek": badge, "fitur_idf": idf, "bobot_tfidf": bobot}

SKEMA = f"""-- =============================================================
-- Basis data Sistem Rekomendasi Coffee Shop Kota Semarang
-- Dihasilkan otomatis oleh 05_data_final.py
-- =============================================================
SET NAMES utf8mb4;
CREATE DATABASE IF NOT EXISTS {NAMA_DB} CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;
USE {NAMA_DB};
SET FOREIGN_KEY_CHECKS = 0;
DROP TABLE IF EXISTS bobot_tfidf, fitur_idf, badge_aspek, kalimat_aspek, ulasan, cafe;
SET FOREIGN_KEY_CHECKS = 1;

CREATE TABLE cafe (
  id_cafe                INT PRIMARY KEY,
  nama_cafe              VARCHAR(100) NOT NULL,
  link                   VARCHAR(500),
  latitude               DECIMAL(10,7) NOT NULL,
  longitude              DECIMAL(10,7) NOT NULL,
  alamat                 VARCHAR(255),
  rating_rata2           DECIMAL(2,1) NULL,
  jumlah_ulasan_total    INT NULL,
  jumlah_ulasan_terambil INT NOT NULL DEFAULT 0,
  harga_indikasi         VARCHAR(50),
  kategori_harga         ENUM('murah','sedang','mahal','tidak_diketahui') NOT NULL,
  jam_operasional        VARCHAR(50),
  dokumen_fitur          VARCHAR(255)
) ENGINE=InnoDB;

CREATE TABLE ulasan (
  id_ulasan      INT PRIMARY KEY,
  id_cafe        INT NOT NULL,
  ulasan         TEXT NOT NULL,
  rating_bintang TINYINT NOT NULL,
  INDEX idx_ulasan_cafe (id_cafe),
  FOREIGN KEY (id_cafe) REFERENCES cafe(id_cafe) ON DELETE CASCADE
) ENGINE=InnoDB;

CREATE TABLE kalimat_aspek (
  id_aspek_kalimat INT PRIMARY KEY,
  id_kalimat       INT NOT NULL,
  id_ulasan        INT NOT NULL,
  id_cafe          INT NOT NULL,
  aspek            ENUM('wifi','harga','suasana','pelayanan') NOT NULL,
  sentimen         ENUM('positif','negatif','netral') NOT NULL,
  sumber_label     ENUM('manual','model') NOT NULL,
  kalimat_asli     TEXT NOT NULL,
  INDEX idx_kalimat_cafe_aspek (id_cafe, aspek, sentimen),
  FOREIGN KEY (id_cafe) REFERENCES cafe(id_cafe) ON DELETE CASCADE,
  FOREIGN KEY (id_ulasan) REFERENCES ulasan(id_ulasan) ON DELETE CASCADE
) ENGINE=InnoDB;

CREATE TABLE badge_aspek (
  id_cafe        INT NOT NULL,
  aspek          ENUM('wifi','harga','suasana','pelayanan') NOT NULL,
  n_positif      INT NOT NULL,
  n_negatif      INT NOT NULL,
  n_netral       INT NOT NULL,
  n_total        INT NOT NULL,
  persen_positif DECIMAL(5,1) NOT NULL,
  sentimen_badge VARCHAR(10) NOT NULL DEFAULT '',
  badge          VARCHAR(50) NOT NULL DEFAULT '',
  PRIMARY KEY (id_cafe, aspek),
  FOREIGN KEY (id_cafe) REFERENCES cafe(id_cafe) ON DELETE CASCADE
) ENGINE=InnoDB;

CREATE TABLE fitur_idf (
  fitur VARCHAR(30) PRIMARY KEY,
  df    INT NOT NULL,
  idf   DECIMAL(10,6) NOT NULL
) ENGINE=InnoDB;

CREATE TABLE bobot_tfidf (
  id_cafe      INT NOT NULL,
  fitur        VARCHAR(30) NOT NULL,
  tf           DECIMAL(6,4) NOT NULL,
  idf          DECIMAL(10,6) NOT NULL,
  bobot        DECIMAL(12,6) NOT NULL,
  bobot_normal DECIMAL(10,6) NOT NULL,
  PRIMARY KEY (id_cafe, fitur),
  INDEX idx_bobot_fitur (fitur),
  FOREIGN KEY (id_cafe) REFERENCES cafe(id_cafe) ON DELETE CASCADE,
  FOREIGN KEY (fitur) REFERENCES fitur_idf(fitur)
) ENGINE=InnoDB;
"""

def nilai_sql(v):
    if v is None or (isinstance(v, float) and math.isnan(v)) or v is pd.NA:
        return "NULL"
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return repr(round(v, 7)) if isinstance(v, float) else str(v)
    s = str(v).replace("\\", "\\\\").replace("'", "\\'").replace("\n", "\\n").replace("\r", "\\r")
    return f"'{s}'"

def main():
    tabel = siapkan_tabel()
    os.makedirs(FOLDER, exist_ok=True)
    urutan = ["cafe", "ulasan", "kalimat_aspek", "badge_aspek", "fitur_idf", "bobot_tfidf"]
    with open(f"{NAMA_DB}.sql", "w", encoding="utf-8") as f:
        f.write(SKEMA + "\nSET FOREIGN_KEY_CHECKS = 0;\n")
        for nama in urutan:
            df = tabel[nama].astype(object).where(tabel[nama].notna(), None)
            df.to_csv(f"{FOLDER}/{nama}.csv", index=False)
            kolom = ", ".join(df.columns)
            nilai = [f"({', '.join(nilai_sql(v) for v in row)})" for row in df.itertuples(index=False)]
            f.write(f"\n-- {nama}: {len(df)} baris\n")
            for i in range(0, len(nilai), 500):
                f.write(f"INSERT INTO {nama} ({kolom}) VALUES\n" + ",\n".join(nilai[i:i + 500]) + ";\n")
        f.write("\nSET FOREIGN_KEY_CHECKS = 1;\n")

    print("=== Data final ===")
    for nama in urutan:
        print(f"{nama:<14}: {len(tabel[nama]):>6} baris")
    print(f"\nFile SQL: {NAMA_DB}.sql ({os.path.getsize(NAMA_DB + '.sql') / 1e6:.1f} MB)")

if __name__ == "__main__":
    main()

=== Data final ===
cafe          :    440 baris
ulasan        :  71483 baris
kalimat_aspek :  58643 baris
badge_aspek   :   1421 baris
fitur_idf     :     10 baris
bobot_tfidf   :   1487 baris

File SQL: db_coffee_shop.sql (16.9 MB)


In [1]:
# =============================================================
# 06_buat_penilaian.py
# Tahap 3.7.2 Evaluasi Relevansi Rekomendasi (bagian 1)
#   Menyusun file penilaian manual:
#   10 skenario preferensi x (10 rekomendasi teratas + 10 coffee shop
#   acak dari peringkat 11 ke bawah), diacak tanpa menampilkan
#   peringkat/skor agar penilaian tidak bias.
# Input : cafe_praproses.csv, aspek_kalimat.csv, bobot_tfidf.csv, idf_fitur.csv
# Output: penilaian_rekomendasi.xlsx (diisi penilai), kunci_penilaian.csv (jangan dibuka saat menilai)
# =============================================================
import math
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

SEED, TOP_N, N_ACAK, N_BUKTI = 42, 10, 10, 3
ASPEK_POSITIF = ["wifi_positif", "harga_positif", "suasana_positif", "pelayanan_positif"]
NAMA_ASPEK = {"wifi": "wifi kencang/stabil", "harga": "harga terjangkau/sepadan",
              "suasana": "suasana nyaman", "pelayanan": "pelayanan ramah/baik"}
NAMA_BUDGET = {"murah": "murah (sampai Rp25.000)", "sedang": "sedang (Rp25.001 sampai Rp50.000)",
               "mahal": "mahal (di atas Rp50.000)"}
SKENARIO = [
    (1, "murah", ["wifi"]), (2, "murah", ["suasana"]), (3, "murah", ["wifi", "suasana"]),
    (4, "murah", ["pelayanan"]), (5, "sedang", ["wifi"]), (6, "sedang", ["suasana", "pelayanan"]),
    (7, "sedang", ["harga"]), (8, "mahal", ["suasana"]), (9, "mahal", ["pelayanan"]),
    (10, "mahal", ["wifi", "suasana"]),
]

def peringkat(bobot, idf, budget, prioritas):
    """Cosine similarity, sama persis dengan 04_dokumen_fitur_cbf.py dan PHP."""
    dipilih = [f"budget_{budget}"] + [f"{a}_positif" for a in prioritas]
    tf_q = {f: 1.0 for f in ASPEK_POSITIF}
    tf_q.update({f: 2.0 for f in dipilih})
    q = {f: t * idf.get(f, 0.0) for f, t in tf_q.items()}
    n = math.sqrt(sum(v ** 2 for v in q.values()))
    q = {f: v / n for f, v in q.items()}
    return (bobot[bobot["fitur"].isin(q)]
            .assign(k=lambda d: d["bobot_normal"] * d["fitur"].map(q))
            .groupby("id_cafe")["k"].sum().sort_values(ascending=False))

def main():
    cafe = pd.read_csv("cafe_praproses.csv")
    cafe.loc[cafe["harga_batas_atas"].isna(), "harga_indikasi"] = "Tidak diketahui"
    kalimat = pd.read_csv("aspek_kalimat.csv", keep_default_na=False)
    bobot = pd.read_csv("bobot_tfidf.csv")
    idf = dict(pd.read_csv("idf_fitur.csv")[["fitur", "idf"]].values)
    info = cafe.set_index("id_cafe")

    baris, kunci = [], []
    for sid, budget, prioritas in SKENARIO:
        skor = peringkat(bobot, idf, budget, prioritas)
        top = list(skor.index[:TOP_N])
        pool = cafe[(~cafe["id_cafe"].isin(top)) & (cafe["jumlah_ulasan_terambil"] > 0)]
        acak = list(pool.sample(N_ACAK, random_state=SEED + sid)["id_cafe"])
        rank = {c: i + 1 for i, c in enumerate(skor.index)}
        teks_pref = f"Budget {NAMA_BUDGET[budget]}; prioritas: " + ", ".join(NAMA_ASPEK[a] for a in prioritas)
        kandidat = [(c, 1) for c in top] + [(c, 0) for c in acak]
        kandidat = pd.DataFrame(kandidat, columns=["id_cafe", "direkomendasikan"]).sample(frac=1, random_state=SEED + sid)
        for c, rek in kandidat.itertuples(index=False):
            bukti = []
            for a in prioritas:
                kal = kalimat[(kalimat["id_cafe"] == c) & (kalimat["aspek"] == a)]
                contoh = kal.sample(min(N_BUKTI, len(kal)), random_state=SEED) if len(kal) else kal
                bukti.append(f"[{a.upper()}] dibahas dalam {len(kal)} kalimat ulasan")
                bukti += [f"  - {t}" for t in contoh["kalimat_asli"]] or ["  - (tidak ada ulasan yang membahas aspek ini)"]
            baris.append({
                "id_skenario": sid, "preferensi": teks_pref, "nama_cafe": info.at[c, "nama_cafe"],
                "harga_indikasi": info.at[c, "harga_indikasi"],
                "rating": info.at[c, "rating_rata2"], "bukti_ulasan": "\n".join(bukti),
                "link_google_maps": info.at[c, "link"],
            })
            kunci.append({"id_skenario": sid, "id_cafe": c, "direkomendasikan": rek,
                          "peringkat": rank.get(c, ""), "cosine": round(float(skor.get(c, 0.0)), 4)})

    df = pd.DataFrame(baris)
    df.insert(0, "no", range(1, len(df) + 1))
    kunci = pd.DataFrame(kunci)
    kunci.insert(0, "no", range(1, len(kunci) + 1))
    kunci.to_csv("kunci_penilaian.csv", index=False)

    wb = Workbook()
    ws = wb.active
    ws.title = "penilaian"
    ws.append(list(df.columns) + ["relevan"])
    for r in df.itertuples(index=False):
        ws.append(list(r) + [""])
    for c in ws[1]:
        c.font = Font(bold=True)
        c.fill = PatternFill("solid", fgColor="D9D9D9")
    for col, w in {"A": 5, "B": 10, "C": 32, "D": 26, "E": 16, "F": 7, "G": 80, "H": 18, "I": 10}.items():
        ws.column_dimensions[col].width = w
    for row in ws.iter_rows(min_row=2):
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
    dv = DataValidation(type="list", formula1='"ya,tidak"', allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(f"I2:I{ws.max_row}")
    ws.freeze_panes = "A2"

    pd_ = wb.create_sheet("panduan")
    for b in [
        ["PANDUAN PENILAIAN RELEVANSI"],
        ["Isi kolom 'relevan' dengan 'ya' atau 'tidak' untuk setiap baris, sesuai preferensi pada kolom 'preferensi'."],
        ["ya", "Harga indikasi sesuai budget DAN bukti ulasan menunjukkan aspek prioritas umumnya dinilai baik."],
        ["tidak", "Harga tidak sesuai budget, ATAU harga tidak diketahui, ATAU aspek prioritas tidak dibahas, ATAU ulasannya cenderung buruk."],
        ["Catatan", "Nilai setiap baris secara mandiri. Urutan baris sudah diacak dan tidak menunjukkan peringkat sistem."],
        ["Catatan", "Bila bukti di kolom ulasan kurang meyakinkan, buka link Google Maps untuk membaca ulasan lain."],
        ["Penting", "Jangan membuka file kunci_penilaian.csv sebelum seluruh penilaian selesai."],
    ]:
        pd_.append(b)
    pd_["A1"].font = Font(bold=True)
    pd_.column_dimensions["A"].width = 10
    pd_.column_dimensions["B"].width = 120
    wb.save("penilaian_rekomendasi.xlsx")

    print("Jumlah baris penilaian:", len(df))
    print(kunci.groupby("id_skenario")["direkomendasikan"].sum().rename("jumlah_rekomendasi").to_string())

if __name__ == "__main__":
    main()

Jumlah baris penilaian: 200
id_skenario
1     10
2     10
3     10
4     10
5     10
6     10
7     10
8     10
9     10
10    10


In [2]:
# =============================================================
# 07_hitung_evaluasi.py
# Tahap 3.7.2 Evaluasi Relevansi Rekomendasi (bagian 2)
#   Menghitung precision, recall, dan F1-score per skenario dari
#   hasil penilaian manual, lalu rata-ratanya (bahan BAB 4.7.2).
#   TP = direkomendasikan & relevan, FP = direkomendasikan & tidak relevan,
#   FN = tidak direkomendasikan & relevan (pada 20 coffee shop yang dinilai)
# Input : penilaian_rekomendasi.xlsx (sudah diisi), kunci_penilaian.csv
# Output: hasil_evaluasi_rekomendasi.csv
# =============================================================
import pandas as pd

def main():
    nilai = pd.read_excel("penilaian_rekomendasi.xlsx", sheet_name="penilaian")[["no", "relevan"]]
    nilai["relevan"] = nilai["relevan"].astype(str).str.strip().str.lower()
    kosong = (~nilai["relevan"].isin(["ya", "tidak"])).sum()
    if kosong:
        print(f"Peringatan: {kosong} baris belum dinilai dan tidak dihitung.")
    data = pd.read_csv("kunci_penilaian.csv").merge(nilai, on="no")
    data = data[data["relevan"].isin(["ya", "tidak"])]
    data["rel"] = (data["relevan"] == "ya").astype(int)

    hasil = []
    for sid, g in data.groupby("id_skenario"):
        tp = int(((g["direkomendasikan"] == 1) & (g["rel"] == 1)).sum())
        fp = int(((g["direkomendasikan"] == 1) & (g["rel"] == 0)).sum())
        fn = int(((g["direkomendasikan"] == 0) & (g["rel"] == 1)).sum())
        tn = int(((g["direkomendasikan"] == 0) & (g["rel"] == 0)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * p * r / (p + r) if p + r else 0.0
        hasil.append({"id_skenario": sid, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                      "precision": round(p, 4), "recall": round(r, 4), "f1_score": round(f1, 4)})
    hasil = pd.DataFrame(hasil)
    rata = hasil[["precision", "recall", "f1_score"]].mean().round(4)
    hasil.to_csv("hasil_evaluasi_rekomendasi.csv", index=False)

    print("=== Evaluasi relevansi rekomendasi per skenario ===")
    print(hasil.to_string(index=False))
    print("\nRata-rata precision :", rata["precision"])
    print("Rata-rata recall    :", rata["recall"])
    print("Rata-rata F1-score  :", rata["f1_score"])

if __name__ == "__main__":
    main()

=== Evaluasi relevansi rekomendasi per skenario ===
 id_skenario  TP  FP  FN  TN  precision  recall  f1_score
           1  10   0   0  10        1.0  1.0000    1.0000
           2  10   0   0  10        1.0  1.0000    1.0000
           3  10   0   2   8        1.0  0.8333    0.9091
           4   7   3   1   9        0.7  0.8750    0.7778
           5  10   0   2   8        1.0  0.8333    0.9091
           6   8   2   4   6        0.8  0.6667    0.7273
           7  10   0   4   6        1.0  0.7143    0.8333
           8  10   0   0  10        1.0  1.0000    1.0000
           9  10   0   0  10        1.0  1.0000    1.0000
          10  10   0   1   9        1.0  0.9091    0.9524

Rata-rata precision : 0.95
Rata-rata recall    : 0.8832
Rata-rata F1-score  : 0.9109
